# Modelos de efeitos fixos H1 (Tese Fernando)

- Quinta tentativa, após reuniões com Professores Nicolella e Sakurai (17 e 18/11/2026);
- Análise passo a passo, partindo de OLS e variáveis passo a passo.

In [1]:
## Diagnóstico do Modelo Inicial e Linha de Base (Etapas 1 a 3)

# **Decisões metodológicas documentadas (grill-me e `PRE_ESPECIFICACAO.md`):**
# 1. **Amostra observada (`obs = TRUE`):** o dataframe completo (`df`, 3.384 linhas) contém 612 observações (18,1%) cujo desfecho `y1a_taxa_conversao__*` foi preenchido na base precária (50% interpolação linear municipal + 50% mediana da microrregião×ano). Nos modelos a seguir, restringimos a estimação principal às **2.772 observações efetivamente observadas (137 municípios)**.
# 2. **Tratamento de `multas_ibama`:** na base precária, 48% das linhas de `x3_n_autos_flora__ibama_espacial` receberam zero administrativo. Conforme o texto do artigo (Seção 2.3), ausências não equivalem a zero real. Por isso, retiramos os autos do modelo principal e os avaliamos: (i) na amostra fiscal elegível (`fisc = TRUE`, 1.140 obs, Seção 2.4) e (ii) por meio dos **embargos do IBAMA geocodificados** na amostra completa (Caminho B2, Seção 2.7).
# 3. **Padronização e escala (`ESCOLHENDO VARIAVEIS H1 PREC.txt`):**
#    - Desfecho em pontos percentuais: `y_agr = 100 * y1a_taxa_conversao__agricultura` e `y_pas = 100 * y1a_taxa_conversao__pastagem`.
#    - Preços reais em log natural centrados na média dos 24 anos (`lps`, `lpm`).
#    - Aptidão GAEZ padronizada pelo desvio-padrão populacional dos 141 municípios (`zs`, `zm`).
#    - Interações principais: `xs = lps * zs` e `xm = lpm * zm`.
#    - Controle climático: `rain = x1c_v2__precip_anomaly_1981_2000 / 100` (em 100 mm).

### 0. AMBIENTE

#### 0.1 Bibliotecas e ambiente

In [2]:
## Carregar pacotes
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(fixest))
suppressPackageStartupMessages(library(sandwich))
suppressPackageStartupMessages(library(lmtest))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(sf))

In [3]:
# Exibe todas as colunas das tabelas, sem saída LaTeX.
options(
    width = 150, repr.matrix.max.cols = Inf,
    jupyter.display_mimetypes = setdiff(getOption("jupyter.display_mimetypes"), "text/latex")
)

# Configura a execução do fixest para esta análise.
setFixest_nthreads(1)
setFixest_notes(FALSE)

# Ativa as operações geográficas com s2.
sf_use_s2(TRUE)

In [4]:
# Indica a pasta local do notebook e da base principal.
dir_modelos <- "C:/Users/ferna/Documents/dev/Tese USP/tese_eco_usp/2_INTI/chapters/chap1/stats/2. MODELOS/grupos_h1/2_base_completa_h1_precaria/MODELOS_H1_PREC"

# Usa a pasta do notebook se a base não estiver no diretório atual. Aponta a base:
if (!file.exists("base_modelos_h1_prec.csv")) setwd(dir_modelos)

# Localiza a pasta 2_INTI a partir do diretório do modelo.
root <- normalizePath(file.path(getwd(), "../../../../../../.."), winslash = "/")

# Confere as condições necessárias para seguir com a análise.
stopifnot(basename(root) == "2_INTI")

# Separa os arquivos exportados pela versão 3.
out_dir <- file.path(root, "audit", "modelos_h1_prec_v3_20261118", "saidas")

# Cria a pasta de saída da versão 3.
dir.create(out_dir, showWarnings = FALSE, recursive = TRUE)

#### 0.2 Leitura e preparação da base

In [5]:
# Lê indicadores lógicos do CSV.
tf <- function(x) tolower(as.character(x)) == "true"

# Define o desvio-padrão populacional para padronizar aptidões.
sd_pop <- function(x) sqrt(mean((x - mean(x, na.rm = TRUE))^2, na.rm = TRUE))

# Lê a mesma base municipal usada na versão 2.
raw_v2 <- as_tibble(read.csv("base_modelos_h1_prec.csv", stringsAsFactors = FALSE))

# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(raw_v2) == 3384, !anyDuplicated(raw_v2[c("z0_codigo_ibge", "z0_ano")]))

In [6]:
# Renomear colunas da base (diminuir nome das colunas para facilitar a manipulação)
df2 <- raw_v2 %>% transmute(
  ibge = factor(z0_codigo_ibge),                                 # Código IBGE (fator)
  ibge_int = as.integer(z0_codigo_ibge),                         # Código IBGE (inteiro)
  ano = as.integer(z0_ano),                                      # Ano (inteiro)
  micro = factor(z0_microrregiao),                               # Microrregião (fator)
  bioma = factor(z0_bioma_predominante),                         # Bioma predominante (fator)
  amz = z0_biome_amazonia_share_i,                               # Prop. área municipal Amazônia (numérico)
  cer = z0_biome_cerrado_share_i,                                # Prop. área municipal Cerrado (numérico)
  pan = z0_biome_pantanal_share_i,                               # Prop. área municipal Pantanal (numérico)
  taxa_agr = y1a_taxa_conversao__agricultura,                    # Taxa conv agricultura (numérico)
  y_agr = 100 * y1a_taxa_conversao__agricultura,                 # Taxa conversão agricultura pp (numérico)
  y_pas = 100 * y1a_taxa_conversao__pastagem,                    # Taxa conversão pastagem pp (numérico)
  area_soja_ha = x2b_area_plantada_ha__soja,                     # Área plantada soja ha (numérico)
  area_milho_ha = x2b_area_plantada_ha__milho,                   # Área plantada milho ha (numérico)
  ps_l1 = x2a_preco_real_usd2024_media_anual_lag1__soja,         # Preço real médio ano soja, lag1 (numérico)
  pm_l1 = x2a_preco_real_usd2024_media_anual_lag1__milho,        # Preço real médio ano milho, lag1 (numérico)
  ps_t = x2a_preco_real_usd2024_media_anual__soja,               # Preço real médio ano soja ano corrente (numérico)
  pm_t = x2a_preco_real_usd2024_media_anual__milho,              # Preço real médio ano milho ano corrente (numérico)
  dap_l1 = x2a_preco_real_usd2024_media_anual_lag1__dap,         # Preço real médio ano fertilizante DAP, lag1 (numérico)
  apt_s = x2b_capacidade_valor__soja__gaez_yxHr0_soy,            # Aptidão produtiva soja GAEZ (numérico)
  apt_m = x2b_capacidade_valor__milho__gaez_yxHr0_mze,           # Aptidão produtiva milho GAEZ (numérico)
  cst_s = x2d_custo_conab_soja_ogm_safra1_conab_h_usd2024,       # Custo real ha soja 1ª safra, Conab (numérico)
  cst_m = x2d_custo_conab_milho_2a_ogm_safra0_conab_h_usd2024,   # Custo real ha milho 2ª safra, Conab (numérico)
  rain = x1c_v2__precip_anomaly_1981_2000 / 100,                 # Chuva 100 mm (numérico)
  autos_l1 = x3_n_autos_flora_elegiveis_lag1__ibama_espacial,    # Nr. autos flora Ibama lag1 (numérico)
  obs = tf(x4_v2__flag_amostra_conversao_ambos),                 # Indicador amostra conversão (lógico)
  fisc = tf(x4_v2__flag_amostra_fiscal_lag1)                     # Indicador elegibilidade amostra fiscal (lógico)
)

In [7]:
head(df2, 2)

ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc
<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>
5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE
5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE


In [8]:
str(df2)

tibble [3,384 × 26] (S3: tbl_df/tbl/data.frame)
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:3384] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:3384] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:3384] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:3384] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:3384] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:3384] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ area_milho_ha: num [1:3384] 500 300 300 150 200 240 105

#### 0.3 Defasagem dos preços e padronização da aptidão

In [9]:
# anos_v2 tem uma linha por ANO, pois os preços de soja, milho e DAP são comuns aos municípios.
# Cada um dos 24 anos entra uma vez na média dos logaritmos dos preços defasados.
# Essas médias são usadas para centrar os logaritmos dos preços em df2.
# Em ordem cronológica, lead() obtém os preços de soja e milho do ano seguinte.
# Esses preços são incorporados a df2 pelo ano para os testes de placebo.

In [10]:
# Organiza a série anual de preços e suas defasagens.
anos_v2 <- df2 %>% distinct(ano, ps_l1, pm_l1, ps_t, pm_t, dap_l1) %>% arrange(ano) %>%
  mutate(ps_lead = lead(ps_t), pm_lead = lead(pm_t))

In [11]:
head(anos_v2, 2)

ano,ps_l1,pm_l1,ps_t,pm_t,dap_l1,ps_lead,pm_lead
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2001,385.9825,161.3827,346.9711,158.8466,268.9632,370.6660,173.0334
2002,346.9711,158.8466,370.6660,173.0334,254.3193,450.0622,179.7087


In [12]:
str(anos_v2)

tibble [24 × 8] (S3: tbl_df/tbl/data.frame)
 $ ano    : int [1:24] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ ps_l1  : num [1:24] 386 347 371 450 510 ...
 $ pm_l1  : num [1:24] 161 159 173 180 186 ...
 $ ps_t   : num [1:24] 347 371 450 510 441 ...
 $ pm_t   : num [1:24] 159 173 180 186 158 ...
 $ dap_l1 : num [1:24] 269 254 265 293 340 ...
 $ ps_lead: num [1:24] 371 450 510 441 418 ...
 $ pm_lead: num [1:24] 173 180 186 158 190 ...


In [13]:
# mun_v2 tem uma linha por MUNICÍPIO, pois a aptidão GAEZ de soja e milho é fixa no tempo.
# Cada um dos 141 municípios entra uma vez na média e no desvio-padrão populacional da aptidão de cada cultura. Esses valores servem para padronizar as aptidões em df2 antes de construir as interações com os logaritmos centrados dos preços defasados.

In [14]:
# Seleciona as aptidões fixas de cada município.
mun_v2 <- df2 %>% distinct(ibge, ibge_int, apt_s, apt_m)

In [15]:
head(mun_v2, 2)

ibge,ibge_int,apt_s,apt_m
<fct>,<int>,<dbl>,<dbl>
5100102,5100102,3665.637,4604.177
5100201,5100201,2971.610,3796.506


In [16]:
str(mun_v2)

tibble [141 × 4] (S3: tbl_df/tbl/data.frame)
 $ ibge    : Factor w/ 141 levels "5100102","5100201",..: 1 2 3 4 5 6 7 8 9 10 ...
 $ ibge_int: int [1:141] 5100102 5100201 5100250 5100300 5100359 5100409 5100508 5100607 5100805 5101001 ...
 $ apt_s   : num [1:141] 3666 2972 2560 3226 2851 ...
 $ apt_m   : num [1:141] 4604 3797 2894 3468 3298 ...


#### 0.4 Auditoria I

In [17]:
stopifnot(nrow(anos_v2) == 24, nrow(mun_v2) == 141)
# Interrompe a execução se as tabelas não tiverem 24 linhas anuais e 141 linhas municipais.
# A checagem confere as dimensões antes de centrar os preços e padronizar as aptidões.

In [18]:
c_ps <- mean(log(anos_v2$ps_l1))
# Calcula a média dos logaritmos do preço da soja defasado em um ano, com peso igual por ano.
# Em 0.5, subtrai-se c_ps do log do preço defasado para obter lps, usado na interação com a aptidão.

In [19]:
c_pm <- mean(log(anos_v2$pm_l1))
# Calcula a média dos logaritmos do preço do milho defasado em um ano, com peso igual por ano.
# Em 0.5, subtrai-se c_pm do log do preço defasado para obter lpm, usado na interação com a aptidão.

In [20]:
c_dap <- mean(log(anos_v2$dap_l1))
# Calcula a média dos logaritmos do preço do DAP defasado em um ano, com peso igual por ano.
# Em 0.5, c_dap centra o log do preço defasado antes das interações com as aptidões de soja e milho.

#### 0.5 Base ajustada

In [21]:
df2 <- df2 %>%
  left_join(anos_v2 %>% select(ano, ps_lead, pm_lead), by = "ano") %>%
  arrange(ibge, ano) %>%
  group_by(ibge) %>%
  mutate(cst_s_l1 = lag(cst_s), cst_m_l1 = lag(cst_m)) %>%
  ungroup() %>%
  mutate(
    lps = log(ps_l1) - c_ps,
    lpm = log(pm_l1) - c_pm,
    zs = (apt_s - mean(mun_v2$apt_s)) / sd_pop(mun_v2$apt_s),
    zm = (apt_m - mean(mun_v2$apt_m)) / sd_pop(mun_v2$apt_m),
    xs = lps * zs,
    xm = lpm * zm,
    trend = (ano - 2001) / 10,
    zs_tr = zs * trend,
    zm_tr = zm * trend,
    xs_f = (log(ps_lead) - c_ps) * zs,
    xm_f = (log(pm_lead) - c_pm) * zm,
    xs_amz = xs * amz, xs_pan = xs * pan,
    xm_amz = xm * amz, xm_pan = xm * pan,
    x_dap_s = (log(dap_l1) - c_dap) * zs,
    x_dap_m = (log(dap_l1) - c_dap) * zm,
    x_cst_s = (log(cst_s_l1) - mean(log(cst_s_l1), na.rm = TRUE)) * zs,
    x_cst_m = (log(cst_m_l1) - mean(log(cst_m_l1), na.rm = TRUE)) * zm,
    f1 = log1p(autos_l1),
    f1_ps = lps * f1, f1_zs = zs * f1, f1_xs = xs * f1,
    f1_pm = lpm * f1, f1_zm = zm * f1, f1_xm = xm * f1,
    post05 = as.numeric(ano >= 2005),
    post06 = as.numeric(ano >= 2006)
  )
# Anexa a df2, pelo ano, os preços futuros de soja e milho para os testes de placebo.
# Ordena por município e ano para calcular as defasagens dos custos dentro de cada município.
# Centra os logaritmos dos preços e padroniza as aptidões entre municípios para construir xs e xm.
# Combina a aptidão com uma tendência em décadas desde 2001 para admitir trajetórias distintas.
# Interage xs e xm com as frações de Amazônia e Pantanal para examinar diferenças por bioma.
# Combina DAP e custos da Conab com as aptidões para avaliar a sensibilidade às medidas de custo.
# Usa log(1 + autos) para manter os zeros e cria interações para os modelos de fiscalização.
# Marca os anos a partir de 2005 e 2006 para comparar recortes temporais alternativos.

In [22]:
head(df2, 2)

ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE,370.6660,173.0334,NA,NA,-0.3467312,-0.3716491,1.960358,1.67101,-0.6797172,-0.6210295,0.0,0.0000000,0.000000,-0.7590935,-0.5045499,0,0,0,0,-1.105066,-0.9419591,NA,NA,0,0,0,0,0,0,0,0,0
5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE,450.0622,179.7087,690.76,536.61,-0.4532818,-0.3874885,1.960358,1.67101,-0.8885945,-0.6474972,0.1,0.1960358,0.167101,-0.3786187,-0.4412974,0,0,0,0,-1.214815,-1.0355095,-0.58813,-0.6638905,0,0,0,0,0,0,0,0,0


In [23]:
str(df2)

tibble [3,384 × 58] (S3: tbl_df/tbl/data.frame)
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:3384] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:3384] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:3384] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:3384] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:3384] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:3384] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:3384] 0 0 0 0 0 0 0 0 0 0 ...
 $ area_milho_ha: num [1:3384] 500 300 300 150 200 240 105

In [24]:
df2 <- df2 %>% mutate(row_id = row_number()) %>% select(row_id, everything())
# Numera as linhas de df2 e põe row_id na primeira coluna para facilitar a conferência da ordem.

In [25]:
d_obs  <- df2 %>% filter(obs)
# Seleciona as linhas com obs = TRUE para estimar os modelos com conversão observada.

In [26]:
str(d_obs)

tibble [2,772 × 59] (S3: tbl_df/tbl/data.frame)
 $ row_id       : int [1:2772] 1 2 3 4 5 6 7 8 9 10 ...
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 1 1 1 1 1 1 ...
 $ ibge_int     : int [1:2772] 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 5100102 ...
 $ ano          : int [1:2772] 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 19 19 19 19 19 19 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ cer          : num [1:2772] 1 1 1 1 1 1 1 1 1 1 ...
 $ pan          : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:2772] 8.56e-05 2.49e-05 2.94e-06 0.00 2.07e-05 ...
 $ y_agr        : num [1:2772] 0.008563 0.002494 0.000294 0 0.002067 ...
 $ y_pas        : num [1:2772] 1.73 1.18 1.35 1.33 1.05 ...
 $ area_soja_ha : num [1:2772] 0 0 0 0 0 0 0 0 0 0 ...
 $

In [27]:
d_fisc <- df2 %>% filter(fisc)
# Seleciona as linhas com fisc = TRUE para analisar os autos de flora na amostra fiscal elegível.

In [28]:
str(d_fisc)

tibble [1,140 × 59] (S3: tbl_df/tbl/data.frame)
 $ row_id       : int [1:1140] 9 10 23 24 29 32 33 34 35 36 ...
 $ ibge         : Factor w/ 141 levels "5100102","5100201",..: 1 1 1 1 2 2 2 2 2 2 ...
 $ ibge_int     : int [1:1140] 5100102 5100102 5100102 5100102 5100201 5100201 5100201 5100201 5100201 5100201 ...
 $ ano          : int [1:1140] 2009 2010 2023 2024 2005 2008 2009 2010 2011 2012 ...
 $ micro        : Factor w/ 22 levels "Alta Floresta",..: 19 19 19 19 9 9 9 9 9 9 ...
 $ bioma        : Factor w/ 3 levels "Amazônia","Cerrado",..: 2 2 2 2 2 2 2 2 2 2 ...
 $ amz          : num [1:1140] 0 0 0 0 0.000238 ...
 $ cer          : num [1:1140] 1 1 1 1 1 ...
 $ pan          : num [1:1140] 0 0 0 0 0 0 0 0 0 0 ...
 $ taxa_agr     : num [1:1140] 1.97e-05 1.98e-05 0.00 0.00 8.35e-05 ...
 $ y_agr        : num [1:1140] 0.00197 0.00198 0 0 0.00835 ...
 $ y_pas        : num [1:1140] 1.25 1.33 1.95 2.6 2.07 ...
 $ area_soja_ha : num [1:1140] 0 0 0 0 50000 ...
 $ area_milho_ha: num [1:1140] 80 

#### 0.6 Auditoria II

In [29]:
stopifnot(nrow(d_obs) == 2772, n_distinct(d_obs$ibge) == 137,
          nrow(d_fisc) == 1140, n_distinct(d_fisc$ibge) == 128)
# Confere 2.772 linhas e 137 municípios em d_obs e 1.140 linhas e 128 municípios em d_fisc.
# Interrompe a execução se as amostras diferirem dessas dimensões antes da estimação.

In [30]:
bioma_amz <- levels(df2$bioma)[grepl("^Amaz", levels(df2$bioma))]
# Recupera o rótulo da Amazônia entre os níveis de bioma pelo prefixo Amaz.
# Esse rótulo é usado para filtrar os municípios de bioma predominante amazônico.

In [31]:
bioma_cer <- levels(df2$bioma)[grepl("^Cerr", levels(df2$bioma))]
# Recupera o rótulo do Cerrado entre os níveis de bioma pelo prefixo Cerr.
# Esse rótulo é usado para filtrar os municípios de bioma predominante cerrado.

In [32]:
bioma_pan <- levels(df2$bioma)[grepl("^Pant", levels(df2$bioma))]
# Recupera pelo prefixo Pant o rótulo do Pantanal para permitir um recorte por bioma predominante.
# Os recortes por rótulo nas células seguintes usam a Amazônia e o Cerrado.

#### 0.7 Funções de estimação

In [33]:
RES <- list()
# Cria a lista que recebe os resultados das estimações e dos testes ao longo do notebook.
# Ao final, os registros são reunidos em uma tabela para consulta e exportação.

In [34]:
rec <- function(m, id, grupo, descr, amostra = NA_character_) {
  ct <- as.data.frame(coeftable(m))
  r2_t <- tryCatch(as.numeric(r2(m, "r2")), error = function(e) NA_real_)
  r2_w <- tryCatch(as.numeric(r2(m, "wr2")), error = function(e) NA_real_)
  RES[[length(RES) + 1]] <<- tibble(
    id = id, grupo = grupo, descricao = descr, amostra = amostra,
    termo = rownames(ct), estimativa = ct[[1]], erro_padrao = ct[[2]], p = ct[[4]], n = nobs(m),
    r2_total = r2_t, r2_within = r2_w
  )
  invisible(m)
}
# Acrescenta a RES uma linha por termo estimado, com coeficiente, erro-padrão e valor-p.
# Registra o modelo, a amostra, o número de observações e os R² total e within.
# Se o cálculo de um R^2 falhar, registra NA para que os demais resultados sejam guardados.
# Devolve o modelo sem imprimi-lo, para que possa ser atribuído a um objeto e usado nas tabelas.

In [35]:
fit_run <- function(crop, data, y = "y_agr", fe = "ibge + bioma^ano", vcov = ~ibge,
                    trend = TRUE, rain = TRUE, extra = NULL) {
  x   <- if (crop == "soja") "xs" else "xm"
  ztr <- if (crop == "soja") "zs_tr" else "zm_tr"
  rhs <- c(x, if (trend) ztr, if (rain) "rain", extra)
  f <- as.formula(paste(y, "~", paste(rhs, collapse = " + "), "|", fe))
  feols(f, data = data, vcov = vcov)
}
# Monta uma fórmula comum às duas culturas, escolhendo xs para soja e xm para milho.
# Por padrão, inclui a tendência por aptidão e o controle de chuva.
# Estima com feols, com efeitos fixos de município e de bioma por ano por padrão.
# Usa erros-padrão agrupados por município por padrão. Os argumentos permitem variar o modelo.

In [36]:
run <- function(crop, data, id, grupo, descr, amostra, ...) {
  rec(fit_run(crop, data, ...), id, grupo, descr, amostra)
}
# Chama fit_run para estimar e rec para registrar o resultado com a identificação e a amostra.
# Isso mantém o mesmo procedimento de registro quando a especificação muda pelos argumentos.

#### 0.8 Formato das tabelas

In [37]:
# Formata estimativas, erros-padrão e métricas de ajuste para leitura.
tab <- function(..., headers = NULL) {
  ms <- list(...)
  if (is.null(headers)) headers <- paste0("(", seq_along(ms), ")")
  estrelas <- function(p) ifelse(p < 0.01, "***", ifelse(p < 0.05, "**", ifelse(p < 0.10, "*", "")))
  termos <- unique(unlist(lapply(ms, function(m) rownames(coeftable(m)))))
  col <- function(m) {
    ct <- coeftable(m)
    vapply(termos, function(t) {
      if (t %in% rownames(ct)) {
        sprintf("%s%s (%s)", formatC(ct[t, 1], digits = 3, format = "g", flag = "#"),
                estrelas(ct[t, 4]), formatC(ct[t, 2], digits = 3, format = "g", flag = "#"))
      } else ""
    }, character(1))
  }
  out <- as.data.frame(do.call(cbind, lapply(ms, col)), stringsAsFactors = FALSE)
  colnames(out) <- headers; rownames(out) <- termos
  info <- do.call(cbind, lapply(ms, function(m) c(
    `N` = format(nobs(m), big.mark = ""),
    `R2 total` = sprintf("%.4f", as.numeric(r2(m, "r2"))),
    `R2 within` = sprintf("%.4f", as.numeric(r2(m, "wr2"))),
    `Efeitos fixos` = paste(m$fixef_vars, collapse = " + "))))
  colnames(info) <- headers
  print(rbind(out, as.data.frame(info, stringsAsFactors = FALSE)))
  cat("Estimativa (erro-padrão agrupado por município, salvo indicação); * p<0,10  ** p<0,05  *** p<0,01\n")
}

#### 0.9 Validação do Ambiente

In [38]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("Ambiente v2 configurado. Amostra observada:", nrow(d_obs), "obs (", n_distinct(d_obs$ibge), "municípios).\n")

Ambiente v2 configurado. Amostra observada: 2772 obs ( 137 municípios).


In [39]:
# Exibe todas as colunas da base em HTML.
options(repr.matrix.max.cols = Inf)
IRdisplay::display_html(repr::repr_html(head(df2, 2)))

row_id,ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<int>,<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE,370.6660,173.0334,NA,NA,-0.3467312,-0.3716491,1.960358,1.67101,-0.6797172,-0.6210295,0.0,0.0000000,0.000000,-0.7590935,-0.5045499,0,0,0,0,-1.105066,-0.9419591,NA,NA,0,0,0,0,0,0,0,0,0
2,5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE,450.0622,179.7087,690.76,536.61,-0.4532818,-0.3874885,1.960358,1.67101,-0.8885945,-0.6474972,0.1,0.1960358,0.167101,-0.3786187,-0.4412974,0,0,0,0,-1.214815,-1.0355095,-0.58813,-0.6638905,0,0,0,0,0,0,0,0,0


In [40]:
tail(df2)

row_id,ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06
<int>,<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3379,5108956,5108956,2019,Alta Floresta,Amazônia,1,0,0,1.756728e-05,1.756728e-03,2.0317592,1509,0,492.8343,205.4161,452.9066,208.6727,491.3879,2652.098,3027.51,887.4231,673.5731,2.4872583,4,TRUE,TRUE,492.6450,200.4740,957.2221,730.7622,-0.1023504,-0.1303901,-0.7098251,-0.3542104,0.07265085,0.04618554,1.8,-1.277685,-0.6375788,0.07292357,0.05481153,0.07265085,0,0.04618554,0,-0.0276496,-0.01379745,-0.01861968,0.03134137,1.6094379,-0.1647265,-1.1424194,0.1169270,-0.2098548,-0.5700797,0.07433276,1,1
3380,5108956,5108956,2020,Alta Floresta,Amazônia,1,0,0,0.000000e+00,0.000000e+00,0.9765656,3800,0,452.9066,208.6727,492.6450,200.4740,376.2261,2652.098,3027.51,872.6322,682.0184,0.4406802,8,TRUE,TRUE,675.8348,300.5185,887.4231,673.5731,-0.1868375,-0.1146605,-0.7098251,-0.3542104,0.13262192,0.04061394,1.9,-1.348668,-0.6729998,-0.15149470,-0.08858149,0.13262192,0,0.04061394,0,0.1619045,0.08079211,0.03512365,0.06020656,2.1972246,-0.4105239,-1.5596451,0.2914002,-0.2519348,-0.7782799,0.08923794,1,1
3381,5108956,5108956,2021,Alta Floresta,Amazônia,1,0,0,0.000000e+00,0.000000e+00,1.1843772,7692,1835,492.6450,200.4740,675.8348,300.5185,378.4231,2652.098,3027.51,771.6819,716.8644,2.7188127,12,TRUE,TRUE,724.1735,341.6679,872.6322,682.0184,-0.1027346,-0.1547429,-0.7098251,-0.3542104,0.07292357,0.05481153,2.0,-1.419650,-0.7084209,-0.20053100,-0.13403735,0.07292357,0,0.05481153,0,0.1577716,0.07872973,0.04705419,0.05579307,2.5649494,-0.2635090,-1.8206653,0.1870453,-0.3969076,-0.9085318,0.14058881,1,1
3382,5108956,5108956,2022,Alta Floresta,Amazônia,1,0,0,1.387445e-05,1.387445e-03,1.9896267,15394,6157,675.8348,300.5185,724.1735,341.6679,694.0289,2652.098,3027.51,811.3913,1220.7631,-1.7536075,3,TRUE,TRUE,615.8341,260.4299,771.6819,716.8644,0.2134254,0.2500815,-0.7098251,-0.3542104,-0.15149470,-0.08858149,2.1,-1.490633,-0.7438419,-0.08550149,-0.03786724,-0.15149470,0,-0.08858149,0,-0.2727379,-0.13609917,0.13432132,0.03814272,1.3862944,0.2958704,-0.9840265,-0.2100163,0.3466866,-0.4910399,-0.12280002,1,1
3383,5108956,5108956,2023,Alta Floresta,Amazônia,1,0,0,3.139054e-07,3.139054e-05,1.1337298,28548,12000,724.1735,341.6679,615.8341,260.4299,828.5465,2652.098,3027.51,1487.0857,1003.5802,-2.8253336,1,TRUE,TRUE,462.7132,190.6186,811.3913,1220.7631,0.2825076,0.3784116,-0.7098251,-0.3542104,-0.20053100,-0.13403735,2.2,-1.561615,-0.7792630,0.11741629,0.07266726,-0.20053100,0,-0.13403735,0,-0.3984900,-0.19885085,0.09870377,-0.15041934,0.6931472,0.1958194,-0.4920132,-0.1389975,0.2622950,-0.2455200,-0.09290761,1,1
3384,5108956,5108956,2024,Alta Floresta,Amazônia,1,0,0,6.342276e-07,6.342276e-05,0.3692473,33773,12847,615.8341,260.4299,462.7132,190.6186,566.6898,2652.098,3027.51,1252.6020,873.4150,-4.2843223,4,TRUE,TRUE,NA,NA,1487.0857,1003.5802,0.1204543,0.1069061,-0.7098251,-0.3542104,-0.08550149,-0.03786724,2.3,-1.632598,-0.8146840,NA,NA,-0.08550149,0,-0.03786724,0,-0.1288553,-0.06430019,-0.33132473,-0.08102869,1.6094379,0.1938637,-1.1424194,-0.1376093,0.1720587,-0.5700797,-0.06094498,1,1


#### 0.10 AED

In [47]:
ls()

[1] "anos_v2"       "baseline_step" "bioma_amz"     "bioma_cer"     "bioma_pan"     "c_dap"         "c_pm"          "c_ps"          "d_fisc"       
[10] "d_obs"         "df2"           "dir_modelos"   "fit_run"       "mun_v2"        "nb_base"       "out_dir"       "raw_v2"        "rec"          
[19] "RES"           "root"          "run"           "sd_pop"        "tab"           "tf"

In [ ]:
# * nb_base: 1. MODELOS. O primeiro passo usa a base inteira. Os passos 2 a 5 usam filter(nb_base, obs).

# * d_obs: Modelos principais M1, M2 e M3 e a maior parte dos exercícios seguintes. Contém as linhas com obs = TRUE.

# * d_fisc: Modelos da amostra fiscal em 2.4. Contém as linhas com fisc = TRUE.

# * df2: Teste de robustez com o painel completo em 2.4, que inclui as linhas com desfecho preenchido. É também a base de origem de nb_base, d_obs e d_fisc.

In [ ]:
# d_obs e d_fisc são recortes de linhas de df2. d_obs mantém as linhas com obs = TRUE, e d_fisc mantém as linhas com fisc = TRUE. São dois filtros aplicados separadamente a df2.

# nb_base é uma versão ampliada de df2: mantém as 3.384 linhas e acrescenta colunas para a especificação inicial. Na seção 1. MODELOS, o primeiro passo usa nb_base inteira e os passos 2 a 5 aplicam filter(nb_base, obs), selecionando as mesmas linhas de d_obs, mas com as colunas adicionais de nb_base.

***

### 1. MODELOS

In [41]:
### Transição passo a passo do `model_1a` para a especificação padronizada
# As tabelas abaixo mostram o efeito isolado de cada correção sobre o `model_1a` (soja e milho):
# - **(1) original:** replica exatamente o `model_1a` da seção 2.1 (3.384 obs, níveis, Y preenchida);
# - **(2) Y observada:** filtra `obs = TRUE` (2.769 obs ativas; `risco_s` deixa de ser significativo, $p = 0,121$);
# - **(3) sem autos:** retira `autos_l1` e `risco` (48% de zeros administrativos);
# - **(4) escala notas:** passa para $Y$ em pontos percentuais e $\ln P_{t-1}$ centrado $\times$ aptidão padronizada ($Z$);
# - **(5) + rain:** inclui anomalia de precipitação (`rain`).

In [42]:
nb_base <- df2 %>% mutate( taxa = taxa_agr, pot_s = ps_l1 * apt_s, pot_m = pm_l1 * apt_m,
  risco_s = ps_l1 * autos_l1, risco_m = pm_l1 * autos_l1 )
# Cria nb_base a partir de df2 e mantém a taxa de conversão agrícola na escala original.
# Reconstrói, por cultura, os produtos do preço defasado pelos valores de aptidão e de autos.
# Esses termos em nível permitem usar a especificação inicial como referência da comparação.

In [43]:
head(nb_base, 2)

row_id,ibge,ibge_int,ano,micro,bioma,amz,cer,pan,taxa_agr,y_agr,y_pas,area_soja_ha,area_milho_ha,ps_l1,pm_l1,ps_t,pm_t,dap_l1,apt_s,apt_m,cst_s,cst_m,rain,autos_l1,obs,fisc,ps_lead,pm_lead,cst_s_l1,cst_m_l1,lps,lpm,zs,zm,xs,xm,trend,zs_tr,zm_tr,xs_f,xm_f,xs_amz,xs_pan,xm_amz,xm_pan,x_dap_s,x_dap_m,x_cst_s,x_cst_m,f1,f1_ps,f1_zs,f1_xs,f1_pm,f1_zm,f1_xm,post05,post06,taxa,pot_s,pot_m,risco_s,risco_m
<int>,<fct>,<int>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,5100102,5100102,2001,Rosário Oeste,Cerrado,0,1,0,8.563177e-05,0.008563177,1.732317,0,500,385.9825,161.3827,346.9711,158.8466,268.9632,3665.637,4604.177,690.76,536.61,0.8217429,0,TRUE,FALSE,370.6660,173.0334,NA,NA,-0.3467312,-0.3716491,1.960358,1.67101,-0.6797172,-0.6210295,0.0,0.0000000,0.000000,-0.7590935,-0.5045499,0,0,0,0,-1.105066,-0.9419591,NA,NA,0,0,0,0,0,0,0,0,0,8.563177e-05,1414872,743034.5,0,0
2,5100102,5100102,2002,Rosário Oeste,Cerrado,0,1,0,2.493791e-05,0.002493791,1.178126,0,300,346.9711,158.8466,370.6660,173.0334,254.3193,3665.637,4604.177,670.80,407.60,-1.5084732,0,TRUE,FALSE,450.0622,179.7087,690.76,536.61,-0.4532818,-0.3874885,1.960358,1.67101,-0.8885945,-0.6474972,0.1,0.1960358,0.167101,-0.3786187,-0.4412974,0,0,0,0,-1.214815,-1.0355095,-0.58813,-0.6638905,0,0,0,0,0,0,0,0,0,2.493791e-05,1271870,731358.0,0,0


In [44]:
baseline_step <- function(crop) {
  p <- if (crop == "soja") list(pot = "pot_s", risco = "risco_s", x = "xs") else list(pot = "pot_m", risco = "risco_m", x = "xm")
  fm <- function(lhs, rhs) as.formula(paste(lhs, "~", rhs, "| ibge + ano"))
  m <- list(
    "1 original"      = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), nb_base, vcov = ~ibge),
    "2 Y observada"   = feols(fm("taxa", paste(p$pot, "+ autos_l1 +", p$risco)), filter(nb_base, obs), vcov = ~ibge),
    "3 sem autos"     = feols(fm("taxa", p$pot), filter(nb_base, obs), vcov = ~ibge),
    "4 escala notas"  = feols(fm("y_agr", p$x), filter(nb_base, obs), vcov = ~ibge),
    "5 + rain"        = feols(fm("y_agr", paste(p$x, "+ rain")), filter(nb_base, obs), vcov = ~ibge)
  )
  for (i in seq_along(m)) {
    rec(m[[i]], paste0("B", i, "_", crop), "baseline", paste(crop, names(m)[i]), if (i == 1) "todas" else "obs")
  }
  cat("=== Linha de base:", toupper(crop), "===\n")
  tab(m[[1]], m[[2]], m[[3]], m[[4]], m[[5]], headers = names(m))
}
# Estima cinco versões por cultura com efeitos fixos de município e ano e erros agrupados por IBGE.
# Parte do painel completo, restringe à conversão observada e depois retira os termos com autos.
# O quarto passo muda duas escalas juntas, a da taxa de conversão e a do termo preço × aptidão.
# O quinto passo acrescenta a anomalia de precipitação para comparar o modelo com esse controle.
# Registra as cinco estimações em RES e as exibe lado a lado para acompanhar os ajustes.

In [45]:
baseline_step("soja")
# Executa as cinco versões para a soja e reúne as estimativas para comparar os ajustes da cultura.

=== Linha de base: SOJA ===
                          1 original          2 Y observada           3 sem autos   4 escala notas            5 + rain
pot_s         6.07e-09*** (2.13e-09) 6.58e-09*** (2.51e-09) 6.08e-09** (2.39e-09)                                     
autos_l1       -2.21e-05* (1.30e-05)   -2.34e-05 (1.71e-05)                                                           
risco_s        4.81e-08** (2.39e-08)    5.14e-08 (3.29e-08)                                                           
xs                                                                                0.140** (0.0545)    0.136** (0.0532)
rain                                                                                               -0.0139** (0.00671)
N                               3384                   2769                  2769             2769                2769
R2 total                      0.3534                 0.3613                0.3607           0.3621              0.3650
R2 within           

In [46]:
baseline_step("milho")
# Repete a sequência para o milho para verificar como os mesmos ajustes afetam suas estimativas.

=== Linha de base: MILHO ===
                          1 original          2 Y observada            3 sem autos     4 escala notas            5 + rain
pot_m         3.48e-09*** (9.40e-10) 4.15e-09*** (1.28e-09) 4.07e-09*** (1.26e-09)                                       
autos_l1        -4.42e-06 (8.40e-06)   -1.17e-06 (1.02e-05)                                                              
risco_m          3.31e-08 (3.38e-08)    1.86e-08 (4.26e-08)                                                              
xm                                                                                 0.0930*** (0.0280)  0.0903*** (0.0275)
rain                                                                                                  -0.0144** (0.00691)
N                               3384                   2769                   2769               2769                2769
R2 total                      0.3494                 0.3567                 0.3566             0.3574              0.

***

## 2.3 Tabela 1 — Associação entre preços reais, capacidade potencial e conversão para agricultura, 2001–2024

In [ ]:
# **Decisões implementadas:**

# - **M1 (soja) e M2 (milho) separados como especificações principais:** incluem controle de tendência diferencial por aptidão (`zs_tr` / `zm_tr`), anomalia de precipitação (`rain`) e efeitos fixos de município e de **bioma predominante $\times$ ano** (`ibge + bioma^ano`), com erros-padrão agrupados por município.

# - **Duplo papel do bioma:** a coluna categórica (`0_bioma_predominante`) absorve choques anuais específicos de cada bioma (`bioma^ano`) e define subamostras (Amazônia vs. Cerrado), enquanto as colunas contínuas de participação territorial (`0_biome_amazonia_share_i` e `0_biome_pantanal_share_i`, tendo o Cerrado como referência) entram nas interações de moderação (`M1b` e `M2b`), preservando os 51 municípios mistos.

# - **M3 (soja e milho juntos) como robustez:** documenta a colinearidade entre os preços anuais ($r = 0,948$) e entre as aptidões municipais ($r = 0,591$).

### Modelos M1 e M2

In [ ]:
# Estima M1 para soja com a amostra observada.
M1 <- run("soja",  d_obs, "M1", "principal", "M1 soja principal", "obs")

In [ ]:
# Estima M2 para milho com a amostra observada.
M2 <- run("milho", d_obs, "M2", "principal", "M2 milho principal", "obs")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== MODELOS PRINCIPAIS (M1 e M2) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(M1, M2, headers = c("M1 soja", "M2 milho"))

### Moderação e subamostras por bioma

In [ ]:
# Adiciona a moderação por fração territorial de bioma ao modelo de soja.
M1b <- run("soja",  d_obs, "M1b", "moderacao", "M1b soja x share bioma", "obs", extra = c("xs_amz", "xs_pan"))

In [ ]:
# Adiciona a moderação por fração territorial de bioma ao modelo de milho.
M2b <- run("milho", d_obs, "M2b", "moderacao", "M2b milho x share bioma", "obs", extra = c("xm_amz", "xm_pan"))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== MODERAÇÃO CONTÍNUA POR SHARE DE BIOMA (Referência = Cerrado) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(M1b, M2b, headers = c("M1b soja", "M2b milho"))

In [ ]:
# Prepara sub para esta etapa da análise.
sub <- list()

In [ ]:
# Estima separadamente cada cultura na Amazônia e no Cerrado.
for (crop in c("soja", "milho")) {
  for (b in c(bioma_amz, bioma_cer)) {
    d <- filter(d_obs, bioma == b)
    sub[[paste(crop, b)]] <- run(crop, d, paste0("S_", crop, "_", substr(b, 1, 3)), "subamostra",
                                 paste(crop, "subamostra", b), "obs", fe = "ibge + ano")
  }
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== SUBAMOSTRAS POR BIOMA PREDOMINANTE (Amazônia vs. Cerrado) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(sub[[1]], sub[[2]], sub[[3]], sub[[4]], headers = names(sub))

### Modelo conjunto e diagnósticos de colinearidade

In [ ]:
# Estima conjuntamente os gradientes de soja e milho.
M3 <- feols(y_agr ~ xs + xm + zs_tr + zm_tr + rain | ibge + bioma^ano, data = d_obs, vcov = ~ibge)

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(M3, "M3", "robustez_cultura", "M3 soja + milho juntos", "obs")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== M3: SOJA E MILHO JUNTOS ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(M3, headers = "M3 soja + milho")

In [ ]:
# Testa a significância conjunta das duas interações de preço.
w3 <- wald(M3, keep = "^x[sm]$", print = FALSE)

In [ ]:
# Estima ou registra a especificação rs.
rs <- resid(feols(xs ~ 1 | ibge + ano, d_obs))

In [ ]:
# Estima ou registra a especificação rm.
rm <- resid(feols(xm ~ 1 | ibge + ano, d_obs))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("Teste Wald conjunto (xs = xm = 0): F = %.3f (p = %.4f)\n", w3$stat, w3$p))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("Correlações: preços lag1 (24 anos) = %.3f | aptidões (141 mun.) = %.3f | xs-xm within = %.3f\n",
            cor(anos_v2$ps_l1, anos_v2$pm_l1), cor(mun_v2$apt_s, mun_v2$apt_m), cor(rs, rm)))

In [ ]:
# Aplica RES no ponto correspondente da análise.
RES[[length(RES) + 1]] <- tibble(id = "M3_wald", grupo = "robustez_cultura", descricao = "teste conjunto xs=xm=0",
  amostra = "obs", termo = "wald_xs_xm", estimativa = w3$stat, erro_padrao = NA_real_, p = w3$p, n = nobs(M3))

***

## 2.4 Tabelas 5 e A6 — Robustez e inferência, com análises de custos e da amostra fiscal

- **Inferência (Etapa 7):** cluster duplo (`ibge + ano`), cluster por microrregião (22 clusters), FE `ibge + micro^ano`, regressão em 24 pontos anuais (Newey-West HAC lag 2) e *leave-one-year-out* (excluindo 1 ano por vez).
- **Placebo e janelas (Etapa 8):** inclusão do preço futuro ($\ln P_{t+1} \times Z$) e recorte pós-ruptura (2006–2024).
- **Desfecho e amostra (Etapa 9):** winsorização a 99%, $\text{asinh}(10Y)$, exclusão do Pantanal, amostra preenchida (3.384 obs) e conversão para pastagem (`y_pas`) como destino comparativo.
- **Custos na base (Etapa 11) e Autos na amostra fiscal (Etapa 10):** avaliação exploratória de $\ln\text{DAP}_{t-1}\times Z$ e $\ln\text{CONAB}_{t-1}\times Z$, e estimação de M6/M7 nos 1.135 pares município-ano elegíveis da amostra fiscal.

### Inferência alternativa (Tabela 5, coluna 5, e Tabela A6, Painel A)

In [ ]:
# Estima as alternativas de efeitos fixos e agrupamento por cultura.
for (crop in c("soja", "milho")) {
  r1 <- run(crop, d_obs, paste0("R_cl2_", crop), "robustez_inferencia", paste(crop, "cluster municipio+ano"), "obs", vcov = ~ibge + ano)
  r2 <- run(crop, d_obs, paste0("R_clmic_", crop), "robustez_inferencia", paste(crop, "cluster microrregiao"), "obs", vcov = ~micro)
  r3 <- run(crop, d_obs, paste0("R_micano_", crop), "robustez_inferencia", paste(crop, "FE microrregiao x ano"), "obs", fe = "ibge + micro^ano")
  cat("=== Inferência alternativa:", toupper(crop), "===\n")
  tab(r1, r2, r3, headers = paste(crop, c("cluster mun+ano", "cluster micro", "FE micro x ano")))
}

### Regressão anual e exclusão de um ano por vez

In [ ]:
# Define a regressão em 24 pontos anuais com inferência HAC.
two_step <- function(crop) {
  d <- d_obs %>% group_by(ibge) %>% mutate(ydm = y_agr - mean(y_agr)) %>% ungroup()
  d$zv  <- if (crop == "soja") d$zs else d$zm
  d$lpv <- if (crop == "soja") d$lps else d$lpm
  sl <- d %>% group_by(ano) %>% summarise(slope = cov(ydm, zv) / var(zv), lp = first(lpv), .groups = "drop")
  m <- lm(slope ~ lp, data = sl)
  ct <- coeftest(m, vcov. = NeweyWest(m, lag = 2, prewhite = FALSE))
  RES[[length(RES) + 1]] <<- tibble(id = paste0("R_2step_", crop), grupo = "robustez_inferencia",
    descricao = paste(crop, "regressao em 24 pontos (HAC lag 2)"), amostra = "obs",
    termo = "lp", estimativa = ct["lp", 1], erro_padrao = ct["lp", 2], p = ct["lp", 4], n = nrow(sl))
  cat(sprintf("Regressão 24 pontos (%s): coef = %.4f (EP HAC = %.4f, p = %.4f)\n", crop, ct["lp", 1], ct["lp", 2], ct["lp", 4]))
}

In [ ]:
# Define o teste de sensibilidade à exclusão de um ano por vez.
loyo <- function(crop) {
  x <- if (crop == "soja") "xs" else "xm"
  out <- map_dfr(sort(unique(d_obs$ano)), function(a) {
    m <- fit_run(crop, filter(d_obs, ano != a))
    rec(m, paste0("R_loyo_", crop, "_", a), "robustez_loyo", paste(crop, "exclui ano", a), "obs")
    ct <- coeftable(m)
    tibble(ano_excluido = a, estimativa = ct[x, 1], p = ct[x, 4])
  })
  cat(sprintf("Leave-one-year-out (%s): min = %.4f, max = %.4f | positivos = %d/24 | p < 0,10 = %d/24\n",
              crop, min(out$estimativa), max(out$estimativa), sum(out$estimativa > 0), sum(out$p < 0.10)))
}

In [ ]:
# Executa o diagnóstico anual com inferência HAC.
two_step("soja")

In [ ]:
# Executa o diagnóstico anual com inferência HAC.
two_step("milho")

In [ ]:
# Reestima o modelo excluindo um ano de cada vez.
loyo("soja")

In [ ]:
# Reestima o modelo excluindo um ano de cada vez.
loyo("milho")

### Tabela 5 — Destino comparativo pastagem, subjanela, placebo temporal e robustez, 2001–2024

In [ ]:
# Prepara q99 para esta etapa da análise.
q99 <- quantile(d_obs$y_agr, 0.99)

In [ ]:
# Prepara d_y para esta etapa da análise.
d_y <- d_obs %>% mutate(y_w = pmin(y_agr, q99), y_a = asinh(10 * y_agr))

In [ ]:
# Estima os placebos, recortes de período e desfechos alternativos por cultura.
for (crop in c("soja", "milho")) {
  xf <- if (crop == "soja") "xs_f" else "xm_f"
  p1 <- run(crop, d_obs, paste0("R_plac_", crop), "placebo", paste(crop, "placebo preco t+1"), "obs", extra = xf)
  w1 <- run(crop, filter(d_obs, ano >= 2006), paste0("R_w06_", crop), "janela", paste(crop, "2006-2024 com controle de ruptura"), "obs 2006-2024")
  w2 <- run(crop, filter(d_obs, ano >= 2006), paste0("R_w06nt_", crop), "janela", paste(crop, "2006-2024 sem controle de ruptura"), "obs 2006-2024", trend = FALSE)
  w3 <- run(crop, d_obs, paste0("R_nt_", crop), "janela", paste(crop, "2001-2024 sem controle de ruptura"), "obs", trend = FALSE)
  cat("=== Placebo e Janelas:", toupper(crop), "===\n")
  tab(p1, w1, w2, w3, headers = paste(crop, c("placebo t+1", "2006-24 c/ tend.", "2006-24 s/ tend.", "2001-24 s/ tend.")))

  a <- run(crop, d_y, paste0("R_win_", crop), "robustez_y", paste(crop, "Y winsor 99%"), "obs", y = "y_w")
  b <- run(crop, d_y, paste0("R_asinh_", crop), "robustez_y", paste(crop, "asinh(10Y)"), "obs", y = "y_a")
  c <- run(crop, filter(d_obs, bioma != bioma_pan), paste0("R_nopan_", crop), "robustez_amostra", paste(crop, "sem Pantanal"), "obs sem Pantanal")
  d <- run(crop, df2, paste0("R_full_", crop), "robustez_amostra", paste(crop, "Y preenchida (3.384)"), "todas")
  e <- run(crop, d_obs, paste0("R_pas_", crop), "pastagem", paste(crop, "Y = pastagem"), "obs", y = "y_pas")
  cat("=== Desfecho e Amostra:", toupper(crop), "===\n")
  tab(a, b, c, d, e, headers = paste(crop, c("winsor 99%", "asinh", "sem Pantanal", "Y preenchida", "Y pastagem")))
}

### Custos e amostra fiscal

In [ ]:
# Compara os modelos que incluem preços de insumos e custos da CONAB.
for (crop in c("soja", "milho")) {
  dap <- if (crop == "soja") "x_dap_s" else "x_dap_m"
  cst <- if (crop == "soja") "x_cst_s" else "x_cst_m"
  a <- run(crop, d_obs, paste0("C_dap_", crop), "custos", paste(crop, "+ DAP x aptidao"), "obs", extra = dap)
  b <- run(crop, d_obs, paste0("C_cst_", crop), "custos", paste(crop, "+ custo CONAB x aptidao"), "obs", extra = cst)
  c <- run(crop, d_obs, paste0("C_both_", crop), "custos", paste(crop, "+ DAP e custo CONAB"), "obs", extra = c(dap, cst))
  cat("=== Custos da base (DAP e CONAB):", toupper(crop), "===\n")
  tab(a, b, c, headers = paste(crop, c("+ DAP", "+ CONAB", "+ ambos")))
}

In [ ]:
# Estima ou registra a especificação F3.
F3 <- feols(y_agr ~ xs + xm + rain | ibge + ano, d_fisc, vcov = ~ibge)

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(F3, "F_M3", "instituicoes", "M3 na amostra fiscal", "fiscal")

In [ ]:
# Estima ou registra a especificação fit_f.
fit_f <- function(x, extra) feols(as.formula(paste("y_agr ~", paste(c(x, extra, "rain"), collapse = " + "), "| ibge + ano")), d_fisc, vcov = ~ibge)

In [ ]:
# Estima ou registra a especificação F6s.
F6s <- fit_f("xs", "f1")

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(F6s, "F_M6_soja",  "instituicoes", "M6 soja",  "fiscal")

In [ ]:
# Estima ou registra a especificação F7s.
F7s <- fit_f("xs", c("f1", "f1_ps", "f1_zs", "f1_xs"))

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(F7s, "F_M7_soja",  "instituicoes", "M7 soja",  "fiscal")

In [ ]:
# Estima ou registra a especificação F6m.
F6m <- fit_f("xm", "f1")

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(F6m, "F_M6_milho", "instituicoes", "M6 milho", "fiscal")

In [ ]:
# Estima ou registra a especificação F7m.
F7m <- fit_f("xm", c("f1", "f1_pm", "f1_zm", "f1_xm"))

In [ ]:
# Registra o modelo e suas métricas no conjunto de resultados.
rec(F7m, "F_M7_milho", "instituicoes", "M7 milho", "fiscal")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== Autos de Flora na Amostra Fiscal Elegível (fisc = TRUE, N = 1.135) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(F3, F6s, F7s, F6m, F7m, headers = c("M3 fiscal", "M6 soja", "M7 soja", "M6 milho", "M7 milho"))

***

## 2.5 Caminho A — Tecnologia e ocupação prévia (Tabela 2) e quebra de regime (Tabela 5)

Esta seção explora o **Caminho A** aprovado pelo autor, lendo diretamente de `2_INTI/xdata/` sem duplicar arquivos:
1. **A.1 Quebra estrutural de regime (pós-2005 e pós-2006):** substitui a tendência linear (`zs_tr`) por quebras de nível (`aptidão × pós-2005/2006` e `amz × pós-2005/2006`), mostrando que a queda abrupta da conversão nos municípios de baixa aptidão após 2005 absorve integralmente o coeficiente de `xs` e `xm`.
2. **A.2 Cenários alternativos de capacidade (`x1_psoja-pmilho/exposicoes_alternativas.csv`):** compara o rendimento potencial GAEZ em **alto insumo (`yxHr0`)** com o de **baixo insumo (`yxLr0`)**, com os índices de aptidão (`sxHr0`, `sxLr0`) e com a produtividade histórica realizada na PAM em 1991–2000 (`pam_1991_2000_min7`). O efeito positivo ocorre exclusivamente no rendimento potencial de **alto insumo (`yxHr0`)**, coerente com a tecnologia de pacotes químicos/cultivares em Mato Grosso (`Spera2017`).
3. **A.3 Aptidão biofísica (GAEZ, sinal $+$) vs. Ocupação agrícola pré-determinada (MapBiomas 1985–2000, sinal $-$) e mecanismo de intensificação:**
   - Constrói a participação municipal média de agricultura em 1985–2000 (classes 20, 39, 41, 62) e de soja em 2000 (classe 39) a partir de `h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv`.
   - Demonstra que a ocupação agrícola pré-2001 é **ortogonal à aptidão GAEZ** ($r = -0,025$), multiplica por quase 4 o $R^2$ within (para `0,0754`) e tem **sinal negativo (`-0,116***` a `-0,146***`)** sobre a abertura de vegetação nativa, ao mesmo tempo em que se associa positivamente à **expansão da área plantada de milho de 2ª safra (`+2,51***` pp/década e `+0,57*` com preço)** em terras já abertas (`MacedoEtAl2012`; `Braganca2018`).

### Quebras em 2005 e 2006 (Tabela 5, coluna 3)

In [ ]:
# Estima o degrau de aptidão para o ano de corte indicado.
ev <- function(crop, pcol) {
  z <- if (crop == "soja") "zs" else "zm"; x <- if (crop == "soja") "xs" else "xm"
  d <- d_obs %>% mutate(p = .data[[pcol]], e_amz = amz * p, e_z = .data[[z]] * p, e_zamz = .data[[z]] * amz * p)
  m <- feols(as.formula(paste("y_agr ~", x, "+ e_amz + e_z + e_zamz + rain | ibge + ano")), d, vcov = ~ibge)
  rec(m, paste0("E_", pcol, "_", crop), "eventos", paste(crop, "evento", pcol), "obs")
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== CAMINHO A.1: Quebra estrutural pós-2005 e pós-2006 ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(ev("soja", "post05"), ev("soja", "post06"), ev("milho", "post05"), ev("milho", "post06"),
    headers = c("soja pós-2005", "soja pós-2006", "milho pós-2005", "milho pós-2006"))

### Tabela 2 — Cenários de capacidade tecnológica, ocupação agrícola pré-determinada (1985–2000) e área de milho

In [ ]:
# Prepara x1_path para esta etapa da análise.
x1_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x1_psoja-pmilho/exposicoes_alternativas.csv")

In [ ]:
# Lê os cenários alternativos de capacidade e seus rendimentos.
x1_alt <- fread(x1_path, showProgress = FALSE) %>%
  as_tibble() %>%
  select(ibge_int = codigo_ibge, ano, produto, cenario_capacidade, capacidade_z) %>%
  distinct() %>%
  mutate(key = paste0(cenario_capacidade, "_", produto)) %>%
  select(ibge_int, ano, key, capacidade_z) %>%
  pivot_wider(names_from = key, values_from = capacidade_z)

### Ocupação prévia e intensificação do milho (Tabela 2, colunas 4–7)

In [ ]:
# Prepara mb_path para esta etapa da análise.
mb_path <- file.path(root, "xdata/chap1/processed/tabular/h1_h2__mapbiomas_classes__col10_1__municipio_bioma_anual.csv")

In [ ]:
# Calcula as medidas de ocupação no período anterior à análise.
mb_pre <- fread(mb_path, showProgress = FALSE) %>%
  as_tibble() %>%
  filter(ano >= 1985, ano <= 2000) %>%
  group_by(ibge_int = codigo_ibge, ano) %>%
  summarise(
    tot_ha  = sum(area_ha),
    agr_ha  = sum(area_ha[class_id %in% c(20, 39, 41, 62)]),
    soja_ha = sum(area_ha[class_id == 39]),
    pas_ha  = sum(area_ha[class_id == 15]),
    .groups = "drop"
  ) %>%
  group_by(ibge_int) %>%
  summarise(
    area_mun_ha  = first(tot_ha),
    agr_sh_85_00 = mean(agr_ha / tot_ha),
    pas_sh_85_00 = mean(pas_ha / tot_ha),
    soja_sh_2000 = (soja_ha / tot_ha)[ano == 2000],
    .groups = "drop"
  ) %>%
  mutate(
    z_exp_agr  = (agr_sh_85_00 - mean(agr_sh_85_00)) / sd_pop(agr_sh_85_00),
    z_exp_pas  = (pas_sh_85_00 - mean(pas_sh_85_00)) / sd_pop(pas_sh_85_00),
    z_exp_soja = (soja_sh_2000 - mean(soja_sh_2000)) / sd_pop(soja_sh_2000)
  )

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(mb_pre) == 141, !anyNA(mb_pre))

In [ ]:
# Integra capacidade alternativa, histórico de ocupação e área plantada.
d_camA <- d_obs %>%
  left_join(x1_alt, by = c("ibge_int", "ano")) %>%
  left_join(mb_pre, by = "ibge_int") %>%
  mutate(
    # Interações com cenários alternativos de capacidade (soja)
    xs_yxLr  = lps * gaez_yxLr0_soy_soja,  zs_yxLr_tr  = gaez_yxLr0_soy_soja * trend,
    xs_sxHr  = lps * gaez_sxHr0_soy_soja,  zs_sxHr_tr  = gaez_sxHr0_soy_soja * trend,
    xs_sxLr  = lps * gaez_sxLr0_soy_soja,  zs_sxLr_tr  = gaez_sxLr0_soy_soja * trend,
    xs_pam90 = lps * pam_1991_2000_min7_soja, zs_pam90_tr = pam_1991_2000_min7_soja * trend,
    # Interações com ocupação agrícola pré-determinada (1985-2000 e soja 2000)
    x_exp_agr_s  = lps * z_exp_agr,  z_exp_agr_tr  = z_exp_agr * trend,
    x_exp_soja_s = lps * z_exp_soja, z_exp_soja_tr = z_exp_soja * trend,
    x_exp_agr_m  = lpm * z_exp_agr,
    # Área plantada (PAM) como % da área municipal para testar intensificação em terras já abertas
    soja_plant_pct  = 100 * area_soja_ha / area_mun_ha,
    milho_plant_pct = 100 * area_milho_ha / area_mun_ha
  )

In [ ]:
# Prepara CA_yxHr para esta etapa da análise.
CA_yxHr <- M1

In [ ]:
# Estima ou registra a especificação CA_yxLr.
CA_yxLr <- rec(feols(y_agr ~ xs_yxLr + zs_yxLr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_yxLr_soja", "caminho_A", "soja GAEZ baixo insumo yxLr0", "obs")

In [ ]:
# Estima ou registra a especificação CA_sxHr.
CA_sxHr <- rec(feols(y_agr ~ xs_sxHr + zs_sxHr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_sxHr_soja", "caminho_A", "soja GAEZ indice aptidao sxHr0", "obs")

In [ ]:
# Estima ou registra a especificação CA_pam.
CA_pam  <- rec(feols(y_agr ~ xs_pam90 + zs_pam90_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "CA_pam90_soja", "caminho_A", "soja PAM rendimento 1991-2000", "obs PAM>=7a")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== CAMINHO A.2: Comparação entre cenários de capacidade (Soja) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(CA_yxHr, CA_yxLr, CA_sxHr, CA_pam, headers = c("GAEZ yx alto (M1)", "GAEZ yx baixo", "GAEZ sx alto", "PAM 1991-2000"))

In [ ]:
# Estima ou registra a especificação EX1.
EX1 <- rec(feols(y_agr ~ x_exp_agr_s + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_agr_soja", "exposicao_pre", "soja: preco x share agr 1985-2000", "obs")

In [ ]:
# Estima ou registra a especificação EX2.
EX2 <- rec(feols(y_agr ~ x_exp_soja_s + z_exp_soja_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_soja2000", "exposicao_pre", "soja: preco x share soja 2000", "obs")

In [ ]:
# Estima ou registra a especificação EX3.
EX3 <- rec(feols(y_agr ~ xs + x_exp_agr_s + zs_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_ambos_soja", "exposicao_pre", "soja: GAEZ + share agr 1985-2000", "obs")

In [ ]:
# Estima ou registra a especificação EX4.
EX4 <- rec(feols(y_agr ~ xm + x_exp_agr_m + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_ambos_milho", "exposicao_pre", "milho: GAEZ + share agr 1985-2000", "obs")

In [ ]:
# Estima ou registra a especificação EX5.
EX5 <- rec(feols(milho_plant_pct ~ xm + x_exp_agr_m + zm_tr + z_exp_agr_tr + rain | ibge + bioma^ano, d_camA, vcov = ~ibge), "EX_intens_milho", "exposicao_pre", "Y = % area plantada milho (2a safra)", "obs")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("\nCorrelações municipais (141) com aptidão GAEZ soja (zs): share_agr_85_00 = %.3f | share_soja_2000 = %.3f\n",
            cor(mun_v2$apt_s, mb_pre$agr_sh_85_00[match(mun_v2$ibge_int, mb_pre$ibge_int)]),
            cor(mun_v2$apt_s, mb_pre$soja_sh_2000[match(mun_v2$ibge_int, mb_pre$ibge_int)])))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== CAMINHO A.3: Potencial GAEZ (+) vs. Fronteira Agrícola Consolidada em 1985-2000 (-) e Intensificação ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(EX1, EX2, EX3, EX4, EX5, headers = c("Y=conv: agr 85-00", "Y=conv: soja 2000", "Y=conv: GAEZ+agr (S)", "Y=conv: GAEZ+agr (M)", "Y=% área milho"))

***

## 2.6 Tabela 3 — Custos logísticos de acesso a mercados e condicionamento por embargos do IBAMA, 2001–2024 (colunas 1–3)

Construído diretamente da malha oficial `2_INTI/xdata/chap1/raw/vector/ibge/MT_Municipios_2023.zip` (`EPSG:4674`, 141 municípios):
- **Distâncias geodésicas municipais (centroide ao ponto de exportação/escoamento):**
  - Eixo Sul/Sudeste: Porto de Santos (`-23.9619, -46.3042`) e Porto de Paranaguá (`-25.5047, -48.5106`), mediana de `1.536 km`.
  - Eixo Arco Norte (BR-163): terminais hidroviários de Miritituba/Itaituba (`-4.2690, -55.9750`), mediana de `1.124 km`.
  - Ferrovia em Mato Grosso: Terminal Ferroviário de Rondonópolis (`-16.5833, -54.7167`), inaugurado em **2013** (mediana de `439 km`).
- **Variáveis logísticas no painel:**
  - `prox_sul_z`: proximidade padronizada aos portos do Sul (`-z(min(Santos, Paranaguá))`), interagida com `xs`/`xm` (`xs_proxsul` / `xm_proxsul`);
  - `dist_min_t_100km`: distância mínima efetiva ao porto ativo no ano $t$ (em 100 km), igual a `min(Santos, Paranaguá)` até 2013 e `min(Santos, Paranaguá, Miritituba)` a partir de **2014** (entrada em operação comercial dos terminais privados de Miritituba/Barcarena no Arco Norte);
  - `choc_ferrov_13`: `prox_rond_z * (ano >= 2013)` (choque de inauguração do terminal ferroviário de Rondonópolis em 2013).

### Distâncias e infraestrutura

In [ ]:
# Localiza a malha municipal do IBGE usada nas distâncias.
zip_path <- normalizePath(file.path(root, "xdata/chap1/raw/vector/ibge/MT_Municipios_2023.zip"), winslash = "/")

In [ ]:
# Lê a malha dos municípios de Mato Grosso.
shp_mt <- st_read(paste0("/vsizip/", zip_path), quiet = TRUE)

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(shp_mt) == 141)

In [ ]:
# Prepara cent_mt para esta etapa da análise.
cent_mt <- st_centroid(st_geometry(shp_mt))

In [ ]:
# Define as coordenadas dos portos e terminais logísticos.
pts_log <- st_sfc(
  st_point(c(-46.3042, -23.9619)), # Porto de Santos (SP)
  st_point(c(-48.5106, -25.5047)), # Porto de Paranaguá (PR)
  st_point(c(-55.9750,  -4.2690)), # Miritituba / Itaituba (PA - Arco Norte)
  st_point(c(-54.7167, -16.5833)), # Terminal Ferroviário de Rondonópolis (MT - 2013)
  crs = st_crs(shp_mt)
)

In [ ]:
# Prepara dmat para esta etapa da análise.
dmat <- as.numeric(st_distance(cent_mt, pts_log)) / 1000

In [ ]:
# Aplica dim no ponto correspondente da análise.
dim(dmat) <- c(nrow(shp_mt), 4)

In [ ]:
# Calcula as distâncias geodésicas municipais.
dist_mun <- tibble(
  ibge_int      = as.integer(shp_mt$CD_MUN),
  dist_sul_km   = pmin(dmat[, 1], dmat[, 2]),
  dist_mirit_km = dmat[, 3],
  dist_rond_km  = dmat[, 4]
) %>%
  mutate(
    prox_sul_z  = -(dist_sul_km - mean(dist_sul_km)) / sd_pop(dist_sul_km),
    prox_rond_z = -(dist_rond_km - mean(dist_rond_km)) / sd_pop(dist_rond_km)
  )

In [ ]:
# Acrescenta ao painel as medidas de logística ou fiscalização desta etapa.
d_camB <- d_obs %>%
  left_join(dist_mun, by = "ibge_int") %>%
  mutate(
    dist_min_t_100km = ifelse(ano <= 2013, dist_sul_km, pmin(dist_sul_km, dist_mirit_km)) / 100,
    x_ps_proxsul     = lps * prox_sul_z,
    xs_proxsul       = xs * prox_sul_z,
    x_pm_proxsul     = lpm * prox_sul_z,
    xm_proxsul       = xm * prox_sul_z,
    choc_ferrov_13   = prox_rond_z * as.numeric(ano >= 2013)
  )

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("Resumo das distâncias geodésicas municipais (km) nos 141 municípios:\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(summary(dist_mun %>% select(dist_sul_km, dist_mirit_km, dist_rond_km)))

### Estimativas com controles logísticos

In [ ]:
# Estima ou registra a especificação L1_s.
L1_s <- rec(feols(y_agr ~ xs + xs_proxsul + x_ps_proxsul + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_prox_soja", "caminho_B_logistica", "soja x proximidade porto Sul", "obs")

In [ ]:
# Estima ou registra a especificação L2_s.
L2_s <- rec(feols(y_agr ~ xs + dist_min_t_100km + choc_ferrov_13 + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_choq_soja", "caminho_B_logistica", "soja + Arco Norte 2014 e Ferrovia 2013", "obs")

In [ ]:
# Estima ou registra a especificação L1_m.
L1_m <- rec(feols(y_agr ~ xm + xm_proxsul + x_pm_proxsul + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_prox_milho", "caminho_B_logistica", "milho x proximidade porto Sul", "obs")

In [ ]:
# Estima ou registra a especificação L2_m.
L2_m <- rec(feols(y_agr ~ xm + dist_min_t_100km + choc_ferrov_13 + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge), "B1_choq_milho", "caminho_B_logistica", "milho + Arco Norte 2014 e Ferrovia 2013", "obs")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== CAMINHO B1: Modelos com Distância aos Portos, Arco Norte (2014) e Ferrovia de Rondonópolis (2013) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(L1_s, L2_s, L1_m, L2_m, headers = c("Soja × Prox. Sul", "Soja + Choques Log.", "Milho × Prox. Sul", "Milho + Choques Log."))

***

## 2.7 Tabela 3, colunas 4–6 — Embargos do IBAMA e medidas alternativas de fiscalização

**Por que usar os embargos geocodificados em vez de apenas `autos_flora`?**
- `x3_n_autos_flora_elegiveis_lag1__ibama_espacial` exige classificação temática completa na fonte, o que reduz a amostra de 2.772 para 1.140 observações (ou introduz 48% de zeros administrativos).
- Lendo diretamente `2_INTI/xdata/chap1/raw/tabular/ibama/termo_embargo_historico.csv`, filtramos os registros de Mato Grosso não cancelados (`SIT_CANCELADO != "S"`), deduplicamos por termo único (`SEQ_TAD`) mantendo a data original do embargo (`DAT_EMBARGO`) e cruzamos as coordenadas (`NUM_LONGITUDE_TAD`, `NUM_LATITUDE_TAD`) com os polígonos oficiais de `MT_Municipios_2023.zip`:
  - **12.283 termos únicos de embargo** entre 2000 e 2024 em MT (100% com código IBGE válido nos 141 municípios e **90,5% com coordenadas válidas geocodificadas por ponto-no-polígono**).
  - Isso permite estimar a dimensão institucional **em toda a amostra observada (`obs = TRUE`, $N = 2.767$, 137 municípios)**, tanto pelo número de embargos geocodificados em $t-1$ (`emb_esp_l1 = ln(1 + n_emb_esp_{t-1})`) quanto pela **área embargada em hectares em $t-1$** (`emb_area_l1 = ln(1 + area_emb_esp_ha_{t-1})`).
  - *Nota metodológica:* conforme registrado em `2_INTI/xdata/chap1/processed/harmonizacao/x3e_infracoes/pendencias.csv`, a série de embargos tem atualizações cadastrais na fonte; por isso deduplicamos estritamente pelo primeiro registro temporal de cada `SEQ_TAD` não cancelado e comparamos também com `n_autos_desmatamento_lag1` de `x3b_infracoes/instituicoes_espacial.csv`.

### Registros e geocodificação

In [ ]:
# Prepara emb_path para esta etapa da análise.
emb_path <- file.path(root, "xdata/chap1/raw/tabular/ibama/termo_embargo_historico.csv")

In [ ]:
# Carrega os registros históricos de embargo nas colunas usadas.
dt_emb <- fread(
  emb_path, sep = ";",
  select = c("SEQ_TAD", "SIT_CANCELADO", "DAT_EMBARGO", "COD_MUNICIPIO", "UF",
             "NUM_LONGITUDE_TAD", "NUM_LATITUDE_TAD", "QTD_AREA_EMBARGADA"),
  encoding = "UTF-8", showProgress = FALSE
)

In [ ]:
# Prepara dt_mt para esta etapa da análise.
dt_mt <- dt_emb[(UF == "MT" | substr(as.character(COD_MUNICIPIO), 1, 2) == "51") & SIT_CANCELADO != "S"]

In [ ]:
# Aplica dt_mt no ponto correspondente da análise.
dt_mt[, dt := as.Date(substr(DAT_EMBARGO, 1, 10))]

In [ ]:
# Aplica dt_mt no ponto correspondente da análise.
dt_mt[, yr := as.integer(format(dt, "%Y"))]

In [ ]:
# Prepara dt_mt para esta etapa da análise.
dt_mt <- dt_mt[!is.na(yr) & yr >= 2000 & yr <= 2024]

In [ ]:
# Aplica setorder no ponto correspondente da análise.
setorder(dt_mt, SEQ_TAD, dt)

In [ ]:
# Prepara dedup_emb para esta etapa da análise.
dedup_emb <- unique(dt_mt, by = "SEQ_TAD")

In [ ]:
# Aplica dedup_emb no ponto correspondente da análise.
dedup_emb[, area_ha := suppressWarnings(as.numeric(gsub(",", ".", QTD_AREA_EMBARGADA)))]

In [ ]:
# Aplica dedup_emb no ponto correspondente da análise.
dedup_emb[is.na(area_ha) | area_ha < 0, area_ha := 0]

In [ ]:
# Aplica dedup_emb no ponto correspondente da análise.
dedup_emb[, lon := suppressWarnings(as.numeric(gsub(",", ".", NUM_LONGITUDE_TAD)))]

In [ ]:
# Aplica dedup_emb no ponto correspondente da análise.
dedup_emb[, lat := suppressWarnings(as.numeric(gsub(",", ".", NUM_LATITUDE_TAD)))]

In [ ]:
# Prepara pts_valid para esta etapa da análise.
pts_valid <- dedup_emb[!is.na(lon) & !is.na(lat) & lon >= -65 & lon <= -48 & lat >= -20 & lat <= -6]

In [ ]:
# Prepara sf_emb para esta etapa da análise.
sf_emb <- st_as_sf(pts_valid, coords = c("lon", "lat"), crs = st_crs(shp_mt))

In [ ]:
# Prepara joined_emb para esta etapa da análise.
joined_emb <- st_join(sf_emb, shp_mt["CD_MUN"], join = st_within)

In [ ]:
# Aplica st_geometry no ponto correspondente da análise.
st_geometry(joined_emb) <- NULL

In [ ]:
# Prepara grid_emb para esta etapa da análise.
grid_emb <- CJ(ibge_int = dist_mun$ibge_int, yr = 2000:2024)

In [ ]:
# Agrega termos e área embargada por município e ano.
emb_esp <- as.data.table(joined_emb)[!is.na(CD_MUN), .( n_emb_esp = .N, area_emb_esp_ha = sum(area_ha)
), by = .(ibge_int = as.integer(CD_MUN), yr)]

In [ ]:
# Prepara a grade anual de embargos e suas defasagens.
emb_panel <- merge(grid_emb, emb_esp, by = c("ibge_int", "yr"), all.x = TRUE)

In [ ]:
# Aplica emb_panel no ponto correspondente da análise.
emb_panel[is.na(n_emb_esp), `:=`(n_emb_esp = 0L, area_emb_esp_ha = 0)]

In [ ]:
# Aplica setorder no ponto correspondente da análise.
setorder(emb_panel, ibge_int, yr)

In [ ]:
# Aplica emb_panel no ponto correspondente da análise.
emb_panel[, `:=`( emb_esp_l1  = log1p(shift(n_emb_esp, 1L)), emb_area_l1 = log1p(shift(area_emb_esp_ha, 1L))
), by = ibge_int]

### Autos de desmatamento e modelos com embargos

In [ ]:
# Prepara x3b_path para esta etapa da análise.
x3b_path <- file.path(root, "xdata/chap1/processed/harmonizacao/x3b_infracoes/instituicoes_espacial.csv")

In [ ]:
# Lê e agrega os autos de desmatamento na chave município e ano.
x3b_desm <- fread(x3b_path, showProgress = FALSE) %>% as_tibble() %>% transmute(
    ibge_int = as.integer(codigo_ibge), ano = as.integer(ano), f_desm_l1 = log1p(n_autos_desmatamento_lag1) )

In [ ]:
# Acrescenta ao painel as medidas de logística ou fiscalização desta etapa.
d_camB <- d_camB %>%
  left_join(as_tibble(emb_panel) %>% rename(ano = yr), by = c("ibge_int", "ano")) %>%
  left_join(x3b_desm, by = c("ibge_int", "ano")) %>%
  mutate(
    emb_ps = lps * emb_esp_l1, emb_zs = zs * emb_esp_l1, emb_xs = xs * emb_esp_l1,
    emb_area_xs = xs * emb_area_l1,
    emb_area_xm = xm * emb_area_l1
  )

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("Termos de embargo únicos não cancelados em MT (2000-2024): %d | Geocodificados na malha 2023: %d (%.1f%%)\n",
            nrow(dedup_emb), nrow(pts_valid), 100 * nrow(pts_valid) / nrow(dedup_emb)))

In [ ]:
# Estima ou registra a especificação B2_s1.
B2_s1 <- rec(feols(y_agr ~ xs + emb_esp_l1 + emb_ps + emb_zs + emb_xs + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_mod_soja", "caminho_B_embargos", "soja + moderacao embargos geocodificados", "obs")

In [ ]:
# Estima ou registra a especificação B2_s2.
B2_s2 <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_xs + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_area_soja", "caminho_B_embargos", "soja + area embargada (ha) lag1", "obs")

In [ ]:
# Estima ou registra a especificação B2_m2.
B2_m2 <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_xm + zm_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
             "B2_emb_area_milho", "caminho_B_embargos", "milho + area embargada (ha) lag1", "obs")

In [ ]:
# Estima ou registra a especificação B2_desm.
B2_desm <- rec(feols(y_agr ~ xs + f_desm_l1 + zs_tr + rain | ibge + bioma^ano, d_camB, vcov = ~ibge),
               "B2_autos_desm_soja", "caminho_B_embargos", "soja + autos desmatamento lag1", "obs c/ desm")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== CAMINHO B2: Embargos do IBAMA Geocodificados (N = 2.767) e Autos de Desmatamento ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(B2_s1, B2_s2, B2_m2, B2_desm,
    headers = c("Soja × N Embargos", "Soja + Área Embarg.", "Milho + Área Embarg.", "Soja + Autos Desm."))

***

## 2.8 Tabela 4 — Sensibilidade da conversão para agricultura aos preços por regime temporal e bioma, 2001–2024

Decompomos a interação principal ($\ln P_{c,t-1} \times Z_{i,c}$) nos três regimes histórico-institucionais destacados no artigo, cruzando-os com os biomas predominantes (**Amazônia** vs. **Cerrado**):
- **Regime P1 (`2001–2005`):** Entrada da China na OMC (dez/2001) e expansão agrícola anterior aos marcos de contenção;
- **Regime P2 (`2006–2011`):** Implementação do PPCDAm e da Moratória da Soja na Amazônia (jul/2006) até a véspera do Novo Código Florestal;
- **Regime P3 (`2012–2024`):** Vigência do Novo Código Florestal (Lei nº 12.651, maio/2012) e expansão logística (Ferrovia de Rondonópolis em 2013 e Arco Norte em 2014).

Também documentamos a decomposição do poder explicativo ($R^2$ total vs. $R^2$ *within*) e as magnitudes econômicas práticas para choques de 1 desvio-padrão e de ciclo completo de preços.

### Interações por regime e bioma (Tabela 4, Painel A)

In [ ]:
# Constrói as interações de preço e aptidão pelos três regimes.
d_reg <- d_obs %>% mutate( d_p1 = as.numeric(ano <= 2005), d_p2 = as.numeric(ano >= 2006 & ano <= 2011),
  d_p3 = as.numeric(ano >= 2012), xs_p1 = xs * d_p1, xs_p2 = xs * d_p2, xs_p3 = xs * d_p3,
  xm_p1 = xm * d_p1, xm_p2 = xm * d_p2, xm_p3 = xm * d_p3 )

In [ ]:
# Estima ou registra a especificação RP_s_mt.
RP_s_mt  <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge),
                'REG_prc_soja_MT', 'regimes_biomas', 'Preços Soja: 3 regimes MT', 'obs')

In [ ]:
# Estima ou registra a especificação RP_s_amz.
RP_s_amz <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge),
                'REG_prc_soja_Amz', 'regimes_biomas', 'Preços Soja: 3 regimes Amazônia', 'obs Amazônia')

In [ ]:
# Estima ou registra a especificação RP_s_cer.
RP_s_cer <- rec(feols(y_agr ~ xs_p1 + xs_p2 + xs_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge),
                'REG_prc_soja_Cer', 'regimes_biomas', 'Preços Soja: 3 regimes Cerrado', 'obs Cerrado')

In [ ]:
# Estima ou registra a especificação RP_m_mt.
RP_m_mt  <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + bioma^ano, d_reg, vcov = ~ibge),
                'REG_prc_milho_MT', 'regimes_biomas', 'Preços Milho: 3 regimes MT', 'obs')

In [ ]:
# Estima ou registra a especificação RP_m_amz.
RP_m_amz <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_amz), vcov = ~ibge),
                'REG_prc_milho_Amz', 'regimes_biomas', 'Preços Milho: 3 regimes Amazônia', 'obs Amazônia')

In [ ]:
# Estima ou registra a especificação RP_m_cer.
RP_m_cer <- rec(feols(y_agr ~ xm_p1 + xm_p2 + xm_p3 + rain | ibge + ano, filter(d_reg, bioma == bioma_cer), vcov = ~ibge),
                'REG_prc_milho_Cer', 'regimes_biomas', 'Preços Milho: 3 regimes Cerrado', 'obs Cerrado')

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat('=== REGIMES TEMPORAIS (2001-05 | 2006-11 | 2012-24) × BIOMAS — H1-PREÇOS ===\n')

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(RP_s_mt, RP_s_amz, RP_s_cer, RP_m_mt, RP_m_amz, RP_m_cer,
    headers = c('Soja MT', 'Soja Amazônia', 'Soja Cerrado', 'Milho MT', 'Milho Amazônia', 'Milho Cerrado'))

### Testes de igualdade entre regimes

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat('\n=== Testes lineares de mudança de regime (P1 vs P2 e P1 vs P3) ===\n')

In [ ]:
# Testa diferenças entre as inclinações dos regimes para cada ajuste.
for (nm in list(list(m = RP_s_mt, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja MT'),
                list(m = RP_s_amz, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja Amazônia'),
                list(m = RP_s_cer, v1 = 'xs_p1', v2 = 'xs_p2', v3 = 'xs_p3', lbl = 'Soja Cerrado'),
                list(m = RP_m_mt, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho MT'),
                list(m = RP_m_amz, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho Amazônia'),
                list(m = RP_m_cer, v1 = 'xm_p1', v2 = 'xm_p2', v3 = 'xm_p3', lbl = 'Milho Cerrado'))) {
  b <- coef(nm$m); V <- vcov(nm$m)
  d12 <- b[nm$v1] - b[nm$v2]; se12 <- sqrt(V[nm$v1, nm$v1] + V[nm$v2, nm$v2] - 2 * V[nm$v1, nm$v2])
  p12 <- 2 * pt(abs(d12 / se12), df = degrees_freedom(nm$m, 't'), lower.tail = FALSE)
  d13 <- b[nm$v1] - b[nm$v3]; se13 <- sqrt(V[nm$v1, nm$v1] + V[nm$v3, nm$v3] - 2 * V[nm$v1, nm$v3])
  p13 <- 2 * pt(abs(d13 / se13), df = degrees_freedom(nm$m, 't'), lower.tail = FALSE)
  cat(sprintf('%-15s | P1-P2 = %+.4f (EP = %.4f, p = %.4f) | P1-P3 = %+.4f (EP = %.4f, p = %.4f)\n',
              nm$lbl, d12, se12, p12, d13, se13, p13))
}

### Médias de conversão e dispersão dos preços

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat('\n=== Médias e desvios-padrão de Y_agr (p.p./ano) por regime na amostra de referência ===\n')

In [ ]:
# Prepara d_ref para esta etapa da análise.
d_ref <- d_reg[fixest::obs(M1), ]

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(d_ref %>% mutate(regime = case_when(ano <= 2005 ~ '1_2001-2005', ano <= 2011 ~ '2_2006-2011', TRUE ~ '3_2012-2024')) %>%
      group_by(regime) %>% summarise(n = n(), media_pp = mean(y_agr), dp_pp = sd(y_agr), .groups = 'drop'))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf('Painel de referência (N=%d): media_pp = %.4f | dp_pp = %.4f\n', nrow(d_ref), mean(d_ref$y_agr), sd(d_ref$y_agr)))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf('Dispersão de ln P (24 anos): DP(ln Ps) = %.4f (amplitude = %.4f) | DP(ln Pm) = %.4f (amplitude = %.4f)\n',
            sd(anos_v2 %>% mutate(lps = log(ps_l1)) %>% pull(lps)), diff(range(log(anos_v2$ps_l1))),
            sd(anos_v2 %>% mutate(lpm = log(pm_l1)) %>% pull(lpm)), diff(range(log(anos_v2$pm_l1)))))

***

## 2.8b Extensões da revisão técnica (Tabelas 4, A5 e A6)

Responde às demandas metodológicas da revisão técnica usando exclusivamente os dados já presentes no projeto (`base_modelos_h1_prec.csv` e `2_INTI/xdata/`):
1. **Amostra efetiva exata ($N = 2.767$), balanço incluídas vs. excluídas e painel balanceado (95 municípios):** estatísticas exatas da amostra efetiva de `M1`/`M2` e reestimação no painel balanceado (`24 anos` completos por município).
2. **Regimes institucionais irrestritos ($Z \times D_{2t}$ e $Z \times D_{3t}$), tendência contínua e teste formal entre biomas:** centragem dos preços dentro de cada regime, inclusão das interações de nível `Z × regime`, testes de Wald conjuntos e contraste direto Amazônia vs. Cerrado (`Amz − Cer`).
3. **Teste formal de diferença entre destinos (`y_agr` vs. `y_pas`) por empilhamento (*stacked*):** estimação empilhada na amostra comum ($N = 2.767 \times 2 = 5.534$) para testar $H_0: \beta_{\text{agr}} = \beta_{\text{past}}$.
4. **Pontes em amostra comum:** (i) GAEZ `yxHr0` vs. PAM 1991–2000 ($N = 1.016$); (ii) amostra fiscal ($N = 1.135$) com especificação alinhada (`ibge + bioma^ano + zs_tr`) e comparação direta com embargos; (iii) decomposição passo a passo de `ev` (tendência $\to$ degrau preservando `bioma^ano`, depois `bioma^ano` $\to$ `ano`, depois interações Amazônia); (iv) ponte CONAB ($N = 2.650$).
5. **Embargos do IBAMA:** hierarquia completa na Eq. (4) (`B2_emb_area_full_*`) com testes de Wald conjuntos e gradiente marginal $\hat{\beta} + \hat{\eta}_2 \ln(1+N)$ com IC 95% pelo método delta na Eq. (5).
6. **Projeção FWL exata com HAC na série anual de *scores* (`DK(2)` e FWL-HAC) e verificação de não-linearidade por tercis de $Z$.**


In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== INICIANDO SEÇÃO 2.8b (ONDA 2) ===\n")

### Amostra efetiva e representatividade

In [ ]:
# Recupera as linhas efetivamente usadas na estimação de M1.
idx_m1 <- fixest::obs(M1)

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(length(idx_m1) == nobs(M1), length(idx_m1) == 2767)

In [ ]:
# Lê áreas de vegetação e conversão para a auditoria amostral.
raw_extra <- raw_v2 %>%
  transmute(
    ibge_int = as.integer(z0_codigo_ibge),
    ano = as.integer(z0_ano),
    nativa_ini_ha = as.numeric(x1b_y_nativa_inicial_ha),
    conv_agr_ha   = as.numeric(y1a_conversao_ha__agricultura),
    conv_pas_ha   = as.numeric(y1a_conversao_ha__pastagem)
  )

In [ ]:
# Prepara eff_keys para esta etapa da análise.
eff_keys <- d_obs[idx_m1, c("ibge_int", "ano")] %>% mutate(in_M1 = TRUE)

In [ ]:
# Identifica a inclusão em M1 e o regime de cada observação.
d_all_diag <- df2 %>% left_join(raw_extra, by = c("ibge_int", "ano")) %>%
  left_join(eff_keys, by = c("ibge_int", "ano")) %>% mutate( in_M1 = replace_na(in_M1, FALSE),
    regime = case_when(ano <= 2005 ~ "1_2001-2005", ano <= 2011 ~ "2_2006-2011", TRUE ~ "3_2012-2024") )

In [ ]:
# Prepara d_eff para esta etapa da análise.
d_eff <- d_all_diag %>% filter(in_M1)

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(d_eff) == 2767)

In [ ]:
# Calcula o resumo geral da amostra efetiva e o resumo por regime.
sum_eff_reg <- bind_rows(
  d_eff %>% mutate(regime = "0_total_M1") %>%
    group_by(regime) %>%
    summarise(
      n = n(), n_mun = n_distinct(ibge_int),
      media_y_agr = mean(y_agr), dp_y_agr = sd(y_agr), mediana_y_agr = median(y_agr),
      media_y_pas = mean(y_pas), media_nativa_ha = mean(nativa_ini_ha),
      razao_somas_pp = 100 * sum(conv_agr_ha) / sum(nativa_ini_ha),
      .groups = "drop"
    ),
  d_eff %>%
    group_by(regime) %>%
    summarise(
      n = n(), n_mun = n_distinct(ibge_int),
      media_y_agr = mean(y_agr), dp_y_agr = sd(y_agr), mediana_y_agr = median(y_agr),
      media_y_pas = mean(y_pas), media_nativa_ha = mean(nativa_ini_ha),
      razao_somas_pp = 100 * sum(conv_agr_ha) / sum(nativa_ini_ha),
      .groups = "drop"
    )
) %>% mutate(pct_n = 100 * n / nrow(d_eff))

In [ ]:
# Calcula a dispersão dos preços em log no período total e por regime.
price_sds <- bind_rows(
  anos_v2 %>% summarise(regime = "0_total_24anos", n_anos = n(),
                        dp_ln_ps = sd(log(ps_l1)), amp_ln_ps = diff(range(log(ps_l1))),
                        dp_ln_pm = sd(log(pm_l1)), amp_ln_pm = diff(range(log(pm_l1)))),
  anos_v2 %>% mutate(regime = case_when(ano <= 2005 ~ "1_2001-2005", ano <= 2011 ~ "2_2006-2011", TRUE ~ "3_2012-2024")) %>%
    group_by(regime) %>%
    summarise(n_anos = n(),
              dp_ln_ps = sd(log(ps_l1)), amp_ln_ps = diff(range(log(ps_l1))),
              dp_ln_pm = sd(log(pm_l1)), amp_ln_pm = diff(range(log(pm_l1))),
              .groups = "drop")
)

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.1: Amostra efetiva exata (N = 2.767) por regime ===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(sum_eff_reg), row.names = FALSE)

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.1: Dispersão dos log-preços (24 anos vs. por regime) ===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(price_sds), row.names = FALSE)

In [ ]:
# Exporta onda2_amostra_regimes.csv para a pasta da versão 3.
write.csv(left_join(sum_eff_reg, price_sds, by = "regime"),
          file.path(out_dir, "onda2_amostra_regimes.csv"), row.names = FALSE, fileEncoding = "UTF-8")

#### Balanço entre observações incluídas e excluídas

In [ ]:
# Prepara bal_vars para esta etapa da análise.
bal_vars <- c("y_agr", "y_pas", "apt_s", "apt_m", "amz", "cer", "pan", "nativa_ini_ha", "area_soja_ha", "area_milho_ha", "rain")

In [ ]:
# Compara as observações incluídas e excluídas da amostra de referência.
bal_tbl <- map_dfr(bal_vars, function(v) {
  x_in  <- d_all_diag[[v]][d_all_diag$in_M1]
  x_out <- d_all_diag[[v]][!d_all_diag$in_M1]
  m_dif <- feols(as.formula(paste(v, "~ in_M1")), data = d_all_diag, vcov = ~ibge_int)
  ct <- coeftable(m_dif)
  tibble(
    variavel = v,
    n_in = sum(!is.na(x_in)), media_in = mean(x_in, na.rm = TRUE), dp_in = sd(x_in, na.rm = TRUE),
    n_out = sum(!is.na(x_out)), media_out = mean(x_out, na.rm = TRUE), dp_out = sd(x_out, na.rm = TRUE),
    diff_in_minus_out = ct["in_M1TRUE", 1], ep_cluster = ct["in_M1TRUE", 2], p_cluster = ct["in_M1TRUE", 4]
  )
})

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.1b: Balanço incluídas (2.767) vs. excluídas (617) ===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(bal_tbl), row.names = FALSE)

In [ ]:
# Exporta onda2_balanco_amostra.csv para a pasta da versão 3.
write.csv(bal_tbl, file.path(out_dir, "onda2_balanco_amostra.csv"), row.names = FALSE, fileEncoding = "UTF-8")

#### Painel balanceado (Tabela A6, Painel A)

In [ ]:
# Identifica os municípios com 24 observações elegíveis.
mun_bal <- d_obs %>% count(ibge_int) %>% filter(n == 24L) %>% pull(ibge_int)

In [ ]:
# Recorta o painel balanceado de municípios elegíveis.
d_bal95 <- d_obs %>% filter(ibge_int %in% mun_bal)

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(d_bal95) == 24L * length(mun_bal))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("\nPainel estritamente balanceado: %d municípios x 24 anos = %d obs.\n", length(mun_bal), nrow(d_bal95)))

In [ ]:
# Estima ou registra a especificação BAL_M1.
BAL_M1 <- run("soja",  d_bal95, "BAL95_M1_soja",  "onda2_balanceado", "M1 soja painel balanceado 95 mun",  "bal95")

In [ ]:
# Estima ou registra a especificação BAL_M2.
BAL_M2 <- run("milho", d_bal95, "BAL95_M2_milho", "onda2_balanceado", "M2 milho painel balanceado 95 mun", "bal95")

In [ ]:
# Inicializa a lista de testes conjuntos e contrastes.
TESTS_O2 <- list()

In [ ]:
# Acrescenta uma linha ao resumo dos testes da Onda 2.
add_test <- function(grupo, teste, cultura = NA_character_, amostra = NA_character_,
                     est = NA_real_, ep = NA_real_, stat = NA_real_, df1 = NA_real_, df2 = NA_real_, p = NA_real_, nota = NA_character_) {
  TESTS_O2[[length(TESTS_O2) + 1]] <<- tibble(
    grupo = grupo, teste = teste, cultura = cultura, amostra = amostra,
    estimativa = est, erro_padrao = ep, estatistica = stat, df1 = df1, df2 = df2, p = p, nota = nota
  )
}

### Regimes irrestritos e diferenças entre biomas (Tabela 4, Painel B)

In [ ]:
# Prepara c_ps_p1 para esta etapa da análise.
c_ps_p1 <- mean(log(anos_v2$ps_l1[anos_v2$ano <= 2005]))

In [ ]:
# Prepara c_ps_p2 para esta etapa da análise.
c_ps_p2 <- mean(log(anos_v2$ps_l1[anos_v2$ano >= 2006 & anos_v2$ano <= 2011]))

In [ ]:
# Prepara c_ps_p3 para esta etapa da análise.
c_ps_p3 <- mean(log(anos_v2$ps_l1[anos_v2$ano >= 2012]))

In [ ]:
# Prepara c_pm_p1 para esta etapa da análise.
c_pm_p1 <- mean(log(anos_v2$pm_l1[anos_v2$ano <= 2005]))

In [ ]:
# Prepara c_pm_p2 para esta etapa da análise.
c_pm_p2 <- mean(log(anos_v2$pm_l1[anos_v2$ano >= 2006 & anos_v2$ano <= 2011]))

In [ ]:
# Prepara c_pm_p3 para esta etapa da análise.
c_pm_p3 <- mean(log(anos_v2$pm_l1[anos_v2$ano >= 2012]))

In [ ]:
# Prepara deslocamentos de nível e preços centrados dentro dos regimes.
mk_reg_vars <- function(d) {
  d %>% mutate(
    d_p1 = as.numeric(ano <= 2005),
    d_p2 = as.numeric(ano >= 2006 & ano <= 2011),
    d_p3 = as.numeric(ano >= 2012),
    lps_reg = log(ps_l1) - case_when(ano <= 2005 ~ c_ps_p1, ano <= 2011 ~ c_ps_p2, TRUE ~ c_ps_p3),
    lpm_reg = log(pm_l1) - case_when(ano <= 2005 ~ c_pm_p1, ano <= 2011 ~ c_pm_p2, TRUE ~ c_pm_p3),
    xs_r_p1 = lps_reg * zs * d_p1, xs_r_p2 = lps_reg * zs * d_p2, xs_r_p3 = lps_reg * zs * d_p3,
    zs_p2   = zs * d_p2,           zs_p3   = zs * d_p3,
    xm_r_p1 = lpm_reg * zm * d_p1, xm_r_p2 = lpm_reg * zm * d_p2, xm_r_p3 = lpm_reg * zm * d_p3,
    zm_p2   = zm * d_p2,           zm_p3   = zm * d_p3
  )
}

In [ ]:
# Prepara d_reg_u para esta etapa da análise.
d_reg_u <- mk_reg_vars(d_obs)

In [ ]:
# Prepara d_bal95_r para esta etapa da análise.
d_bal95_r <- mk_reg_vars(d_bal95)

In [ ]:
# Estima ou registra a especificação RUL_s_mt.
RUL_s_mt  <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                 "REG_lvl_soja_MT", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos MT (Z x regime)", "obs")

In [ ]:
# Estima ou registra a especificação RUL_s_amz.
RUL_s_amz <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_amz), vcov = ~ibge),
                 "REG_lvl_soja_Amz", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos Amazonia", "obs Amazonia")

In [ ]:
# Estima ou registra a especificação RUL_s_cer.
RUL_s_cer <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_cer), vcov = ~ibge),
                 "REG_lvl_soja_Cer", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos Cerrado", "obs Cerrado")

In [ ]:
# Estima ou registra a especificação RUL_m_mt.
RUL_m_mt  <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                 "REG_lvl_milho_MT", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos MT (Z x regime)", "obs")

In [ ]:
# Estima ou registra a especificação RUL_m_amz.
RUL_m_amz <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_amz), vcov = ~ibge),
                 "REG_lvl_milho_Amz", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos Amazonia", "obs Amazonia")

In [ ]:
# Estima ou registra a especificação RUL_m_cer.
RUL_m_cer <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + ano, filter(d_reg_u, bioma == bioma_cer), vcov = ~ibge),
                 "REG_lvl_milho_Cer", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos Cerrado", "obs Cerrado")

In [ ]:
# Estima ou registra a especificação RUT_s_mt.
RUT_s_mt <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + zs_tr + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                "REG_tr_soja_MT", "onda2_regimes_irrestritos", "Soja: 3 regimes irrestritos + zs_tr MT", "obs")

In [ ]:
# Estima ou registra a especificação RUT_m_mt.
RUT_m_mt <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + zm_tr + rain | ibge + bioma^ano, d_reg_u, vcov = ~ibge),
                "REG_tr_milho_MT", "onda2_regimes_irrestritos", "Milho: 3 regimes irrestritos + zm_tr MT", "obs")

In [ ]:
# Estima ou registra a especificação BAL_reg_s.
BAL_reg_s <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain | ibge + bioma^ano, d_bal95_r, vcov = ~ibge),
                 "BAL95_REG_soja", "onda2_balanceado", "Soja: 3 regimes irrestritos bal 95 mun", "bal95")

In [ ]:
# Estima ou registra a especificação BAL_reg_m.
BAL_reg_m <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain | ibge + bioma^ano, d_bal95_r, vcov = ~ibge),
                 "BAL95_REG_milho", "onda2_balanceado", "Milho: 3 regimes irrestritos bal 95 mun", "bal95")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.2: Regimes Institucionais IRRESTRITOS (Z x regime + preços centrados no regime) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(RUL_s_mt, RUL_s_amz, RUL_s_cer, RUL_m_mt, RUL_m_amz, RUL_m_cer,
    headers = c("Soja MT (irrest.)", "Soja Amz", "Soja Cer", "Milho MT (irrest.)", "Milho Amz", "Milho Cer"))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.2b: Painel balanceado 95 mun. (M1/M2 e regimes irrestritos) e regimes + tendência ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(BAL_M1, BAL_M2, RUT_s_mt, RUT_m_mt, BAL_reg_s, BAL_reg_m,
    headers = c("Bal95 M1", "Bal95 M2", "Reg+Tr Soja", "Reg+Tr Milho", "Bal95 Reg S", "Bal95 Reg M"))

#### Testes de nível e de mudança de regime

In [ ]:
# Testa os termos de nível dos regimes e as diferenças entre períodos.
for (item in list(
  list(m = RUL_s_mt,  pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "MT"),
  list(m = RUL_s_amz, pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "Amazonia"),
  list(m = RUL_s_cer, pref = "xs_r", zpat = "^zs_p[23]$", crop = "soja",  am = "Cerrado"),
  list(m = RUL_m_mt,  pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "MT"),
  list(m = RUL_m_amz, pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "Amazonia"),
  list(m = RUL_m_cer, pref = "xm_r", zpat = "^zm_p[23]$", crop = "milho", am = "Cerrado")
)) {
  w <- wald(item$m, keep = item$zpat, print = FALSE)
  add_test("regimes_irrestritos", paste0("Wald Z x regime (", item$am, ")"), item$crop, item$am,
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
  b <- coef(item$m); V <- vcov(item$m); df_t <- degrees_freedom(item$m, "t")
  v1 <- paste0(item$pref, "_p1"); v2 <- paste0(item$pref, "_p2"); v3 <- paste0(item$pref, "_p3")
  for (pair in list(list(lbl = "P1 - P2", vb = v2), list(lbl = "P1 - P3", vb = v3))) {
    dif <- b[v1] - b[pair$vb]
    se_d <- sqrt(V[v1, v1] + V[pair$vb, pair$vb] - 2 * V[v1, pair$vb])
    p_d <- 2 * pt(abs(dif / se_d), df = df_t, lower.tail = FALSE)
    add_test("regimes_irrestritos", paste0(pair$lbl, " (", item$am, ")"), item$crop, item$am,
             est = dif, ep = se_d, stat = dif / se_d, df2 = df_t, p = p_d)
  }
}

#### Contrastes Amazônia e Cerrado

In [ ]:
# Restringe a comparação de biomas e prepara as interações territoriais.
d_ac <- d_reg_u %>%
  filter(bioma %in% c(bioma_amz, bioma_cer)) %>%
  mutate(
    is_amz = as.numeric(bioma == bioma_amz),
    xs_amz_d = xs * is_amz, zs_tr_amz = zs_tr * is_amz,
    xm_amz_d = xm * is_amz, zm_tr_amz = zm_tr * is_amz,
    rain_amz = rain * is_amz,
    xs_p1_amz = xs_r_p1 * is_amz, xs_p2_amz = xs_r_p2 * is_amz, xs_p3_amz = xs_r_p3 * is_amz,
    zs_p2_amz = zs_p2 * is_amz,   zs_p3_amz = zs_p3 * is_amz,
    xm_p1_amz = xm_r_p1 * is_amz, xm_p2_amz = xm_r_p2 * is_amz, xm_p3_amz = xm_r_p3 * is_amz,
    zm_p2_amz = zm_p2 * is_amz,   zm_p3_amz = zm_p3 * is_amz
  )

In [ ]:
# Estima ou registra a especificação BIO_s.
BIO_s <- rec(feols(y_agr ~ xs + xs_amz_d + zs_tr + zs_tr_amz + rain + rain_amz | ibge + bioma^ano, d_ac, vcov = ~ibge),
             "BIO_diff_soja", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado M1 soja", "obs Amz+Cer")

In [ ]:
# Estima ou registra a especificação BIO_m.
BIO_m <- rec(feols(y_agr ~ xm + xm_amz_d + zm_tr + zm_tr_amz + rain + rain_amz | ibge + bioma^ano, d_ac, vcov = ~ibge),
             "BIO_diff_milho", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado M2 milho", "obs Amz+Cer")

In [ ]:
# Estima ou registra a especificação BIO_reg_s.
BIO_reg_s <- rec(feols(y_agr ~ xs_r_p1 + xs_r_p2 + xs_r_p3 + zs_p2 + zs_p3 + rain +
                         xs_p1_amz + xs_p2_amz + xs_p3_amz + zs_p2_amz + zs_p3_amz + rain_amz | ibge + bioma^ano,
                       d_ac, vcov = ~ibge),
                 "REG_diff_soja", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado 3 regimes soja", "obs Amz+Cer")

In [ ]:
# Estima ou registra a especificação BIO_reg_m.
BIO_reg_m <- rec(feols(y_agr ~ xm_r_p1 + xm_r_p2 + xm_r_p3 + zm_p2 + zm_p3 + rain +
                         xm_p1_amz + xm_p2_amz + xm_p3_amz + zm_p2_amz + zm_p3_amz + rain_amz | ibge + bioma^ano,
                       d_ac, vcov = ~ibge),
                 "REG_diff_milho", "onda2_bioma_diff", "Diferenca Amazonia - Cerrado 3 regimes milho", "obs Amz+Cer")

In [ ]:
# Testa em conjunto as diferenças de inclinação entre Amazônia e Cerrado.
for (bpair in list(list(m = BIO_reg_s, pat = "^xs_p[123]_amz$", crop = "soja"),
                   list(m = BIO_reg_m, pat = "^xm_p[123]_amz$", crop = "milho"))) {
  w <- wald(bpair$m, keep = bpair$pat, print = FALSE)
  add_test("bioma_diff", "Wald conjunto Amz - Cer (3 regimes)", bpair$crop, "Amz+Cer",
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.2c: Diferenças formais Amazônia vs. Cerrado ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(BIO_s, BIO_m, BIO_reg_s, BIO_reg_m, headers = c("M1 Amz-Cer", "M2 Amz-Cer", "Reg Soja Amz-Cer", "Reg Milho Amz-Cer"))

### Contraste formal entre agricultura e pastagem (Tabela A6, Painéis A e B)

In [ ]:
# Empilha os destinos agricultura e pastagem na amostra comum.
d_stack <- bind_rows(
  d_eff %>% mutate(dest = "agr", is_agr = 1, y_dest = y_agr),
  d_eff %>% mutate(dest = "pas", is_agr = 0, y_dest = y_pas)
) %>% mutate(
  xs_agr = xs * is_agr, zs_tr_agr = zs_tr * is_agr,
  xm_agr = xm * is_agr, zm_tr_agr = zm_tr * is_agr,
  rain_agr = rain * is_agr
)

In [ ]:
# Estima ou registra a especificação STK_s.
STK_s <- rec(feols(y_dest ~ xs + xs_agr + zs_tr + zs_tr_agr + rain + rain_agr | ibge^dest + bioma^ano^dest,
                   d_stack, vcov = ~ibge),
             "STACK_soja", "onda2_stacked_destinos", "Empilhado Agr vs Past soja", "obs x 2")

In [ ]:
# Estima ou registra a especificação STK_m.
STK_m <- rec(feols(y_dest ~ xm + xm_agr + zm_tr + zm_tr_agr + rain + rain_agr | ibge^dest + bioma^ano^dest,
                   d_stack, vcov = ~ibge),
             "STACK_milho", "onda2_stacked_destinos", "Empilhado Agr vs Past milho", "obs x 2")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.3: Diferença formal Agricultura vs. Pastagem (stacked) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(STK_s, STK_m, headers = c("Soja Agr-Past", "Milho Agr-Past"))

### Comparações em amostras comuns

In [ ]:
# Prepara d_pam_comm para esta etapa da análise.
d_pam_comm <- d_camA[fixest::obs(CA_pam), ]

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(d_pam_comm) == 1016)

In [ ]:
# Estima ou registra a especificação CA_c_gaez.
CA_c_gaez <- rec(feols(y_agr ~ xs + zs_tr + rain | ibge + bioma^ano, d_pam_comm, vcov = ~ibge),
                 "CA_comm_gaez_soja", "onda2_pontes", "Soja GAEZ na amostra PAM (N=1016)", "obs PAM>=7a")

In [ ]:
# Estima ou registra a especificação CA_c_both.
CA_c_both <- rec(feols(y_agr ~ xs + xs_pam90 + zs_tr + zs_pam90_tr + rain | ibge + bioma^ano, d_pam_comm, vcov = ~ibge),
                 "CA_comm_both_soja", "onda2_pontes", "Soja GAEZ + PAM na amostra PAM (N=1016)", "obs PAM>=7a")

In [ ]:
# Calcula o teste de hipótese w_pam.
w_pam <- wald(CA_c_both, keep = "^xs", print = FALSE)

In [ ]:
# Aplica add_test no ponto correspondente da análise.
add_test("pontes", "Wald conjunto GAEZ+PAM (N=1016)", "soja", "obs PAM>=7a",
         stat = w_pam$stat, df1 = w_pam$df1, df2 = w_pam$df2, p = w_pam$p)

#### Autos de Flora e embargos na amostra fiscal comum (Tabela A5, Painel A)

In [ ]:
# Prepara d_fisc_al para esta etapa da análise.
d_fisc_al <- d_camB %>% filter(fisc)

In [ ]:
# Estima ou registra a especificação F_ctrl_s.
F_ctrl_s <- run("soja", d_fisc_al, "F_base_ctrl_soja", "onda2_pontes", "M1 soja na amostra fiscal (ibge + bioma^ano)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_m6_al_s.
F_m6_al_s <- rec(feols(y_agr ~ xs + f1 + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M6_alinhado_soja", "onda2_pontes", "M6 soja alinhado (ibge + bioma^ano + zs_tr)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_m7_al_s.
F_m7_al_s <- rec(feols(y_agr ~ xs + f1 + f1_ps + f1_zs + f1_xs + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M7_alinhado_soja", "onda2_pontes", "M7 soja alinhado (ibge + bioma^ano + zs_tr)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_emb_sub_s.
F_emb_sub_s <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_xs + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                   "F_emb_area_sub_soja", "onda2_pontes", "Embargos area na amostra fiscal (soja)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_ctrl_m.
F_ctrl_m <- run("milho", d_fisc_al, "F_base_ctrl_milho", "onda2_pontes", "M2 milho na amostra fiscal (ibge + bioma^ano)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_m7_al_m.
F_m7_al_m <- rec(feols(y_agr ~ xm + f1 + f1_pm + f1_zm + f1_xm + zm_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                 "F_M7_alinhado_milho", "onda2_pontes", "M7 milho alinhado (ibge + bioma^ano + zm_tr)", "fiscal")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.4a/4b: Pontes PAM (N=1.016) e Fiscal Alinhado (N=1.135) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(CA_c_gaez, CA_pam, CA_c_both, F_ctrl_s, F_m6_al_s, F_m7_al_s, F_emb_sub_s,
    headers = c("GAEZ N=1016", "PAM N=1016", "Ambos N=1016", "M1 fisc", "M6 al.", "M7 al.", "Emb. fisc"))

#### Decomposição do degrau

In [ ]:
# Prepara EV_steps para esta etapa da análise.
EV_steps <- list()

In [ ]:
# Decompõe a troca da tendência linear por degrau e a mudança dos efeitos fixos.
for (crop in c("soja", "milho")) {
  z <- if (crop == "soja") "zs" else "zm"
  x <- if (crop == "soja") "xs" else "xm"
  for (pcol in c("post05", "post06")) {
    d_ev <- d_obs %>% mutate(p = .data[[pcol]], e_z = .data[[z]] * p)
    s1 <- rec(feols(as.formula(paste("y_agr ~", x, "+ e_z + rain | ibge + bioma^ano")), d_ev, vcov = ~ibge),
              paste0("EV_s1_", pcol, "_", crop), "onda2_ev_decomp", paste(crop, pcol, "passo1 degrau c/ bioma^ano"), "obs")
    s2 <- rec(feols(as.formula(paste("y_agr ~", x, "+ e_z + rain | ibge + ano")), d_ev, vcov = ~ibge),
              paste0("EV_s2_", pcol, "_", crop), "onda2_ev_decomp", paste(crop, pcol, "passo2 FE ano"), "obs")
    EV_steps[[paste(crop, pcol, "s1")]] <- s1
    EV_steps[[paste(crop, pcol, "s2")]] <- s2
  }
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.4c: Decomposição passo a passo de `ev` ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(EV_steps[["soja post05 s1"]], EV_steps[["soja post05 s2"]], EV_steps[["milho post05 s1"]], EV_steps[["milho post05 s2"]],
    headers = c("Soja p05 s1 (bioma^ano)", "Soja p05 s2 (ano)", "Milho p05 s1 (bioma^ano)", "Milho p05 s2 (ano)"))

#### Custos CONAB

In [ ]:
# Prepara d_conab para esta etapa da análise.
d_conab <- d_obs %>% filter(!is.na(x_cst_s))

In [ ]:
# Estima ou registra a especificação C_b2650_s.
C_b2650_s <- run("soja",  d_conab, "C_base_2650_soja",  "onda2_pontes", "M1 soja na amostra CONAB (N=2650)",  "obs CONAB")

In [ ]:
# Estima ou registra a especificação C_d2650_s.
C_d2650_s <- run("soja",  d_conab, "C_dap_2650_soja",   "onda2_pontes", "soja + DAP na amostra CONAB (N=2650)", "obs CONAB", extra = "x_dap_s")

In [ ]:
# Estima ou registra a especificação C_b2650_m.
C_b2650_m <- run("milho", d_conab, "C_base_2650_milho", "onda2_pontes", "M2 milho na amostra CONAB (N=2650)", "obs CONAB")

In [ ]:
# Estima ou registra a especificação C_d2650_m.
C_d2650_m <- run("milho", d_conab, "C_dap_2650_milho",  "onda2_pontes", "milho + DAP na amostra CONAB (N=2650)", "obs CONAB", extra = "x_dap_m")

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nobs(C_b2650_s) == 2650, nobs(C_b2650_m) == 2650)

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.4d: Ponte CONAB (N = 2.650) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(C_b2650_s, C_d2650_s, C_b2650_m, C_d2650_m, headers = c("Soja M1 2650", "Soja +DAP 2650", "Milho M2 2650", "Milho +DAP 2650"))

In [ ]:
# Prepara cust_df para esta etapa da análise.
cust_df <- bind_rows(RES) %>% filter(grupo %in% c("principal", "custos") | grepl("^C_(base|dap)_2650", id))

In [ ]:
# Exporta onda2_custos_dap_conab.csv para a pasta da versão 3.
write.csv(cust_df, file.path(out_dir, "onda2_custos_dap_conab.csv"), row.names = FALSE, fileEncoding = "UTF-8")

### Especificações de embargos e gradiente marginal

In [ ]:
# Prepara a hierarquia completa de termos dos modelos de embargo.
d_camB2 <- d_camB %>% mutate( emb_area_ps = lps * emb_area_l1, emb_area_zs = zs * emb_area_l1,
    emb_area_pm = lpm * emb_area_l1, emb_area_zm = zm * emb_area_l1,
    emb_pm = lpm * emb_esp_l1, emb_zm = zm * emb_esp_l1, emb_xm = xm * emb_esp_l1 )

In [ ]:
# Estima ou registra a especificação B2_area_full_s.
B2_area_full_s <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_ps + emb_area_zs + emb_area_xs + zs_tr + rain | ibge + bioma^ano,
                            d_camB2, vcov = ~ibge),
                      "B2_emb_area_full_soja", "onda2_embargos", "soja + area embargada hierarquia completa", "obs")

In [ ]:
# Estima ou registra a especificação B2_area_full_m.
B2_area_full_m <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_pm + emb_area_zm + emb_area_xm + zm_tr + rain | ibge + bioma^ano,
                            d_camB2, vcov = ~ibge),
                      "B2_emb_area_full_milho", "onda2_embargos", "milho + area embargada hierarquia completa", "obs")

In [ ]:
# Estima ou registra a especificação B2_mod_m.
B2_mod_m <- rec(feols(y_agr ~ xm + emb_esp_l1 + emb_pm + emb_zm + emb_xm + zm_tr + rain | ibge + bioma^ano,
                      d_camB2, vcov = ~ibge),
                "B2_emb_mod_milho", "onda2_embargos", "milho + moderacao N embargos hierarquia completa", "obs")

In [ ]:
# Registra os testes conjuntos das interações com embargos.
for (ep in list(
  list(m = B2_s1, pat = "^emb_(ps|zs|xs)$", nm = "Wald interacoes N embargos Eq.5", crop = "soja"),
  list(m = B2_mod_m, pat = "^emb_(pm|zm|xm)$", nm = "Wald interacoes N embargos Eq.5", crop = "milho"),
  list(m = B2_area_full_s, pat = "^emb_area_(ps|zs|xs)$", nm = "Wald interacoes Area embargada Eq.4", crop = "soja"),
  list(m = B2_area_full_m, pat = "^emb_area_(pm|zm|xm)$", nm = "Wald interacoes Area embargada Eq.4", crop = "milho")
)) {
  w <- wald(ep$m, keep = ep$pat, print = FALSE)
  add_test("embargos", ep$nm, ep$crop, "obs", stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
}

In [ ]:
# Prepara d_emb_eff para esta etapa da análise.
d_emb_eff <- d_camB2[idx_m1, ] %>% mutate(n_emb_l1 = round(expm1(emb_esp_l1)))

In [ ]:
# Prepara pts_n para esta etapa da análise.
pts_n <- sort(unique(c(0L, 1L, 2L, 5L, 10L, 20L, 50L, as.integer(quantile(d_emb_eff$n_emb_l1, c(0.5, 0.75, 0.90, 0.95, 1.0))))))

In [ ]:
# Calcula gradiente e intervalo de confiança para cada contagem de embargos.
calc_grad <- function(m, bvar, evar, crop_lbl) {
  b <- coef(m); V <- vcov(m); df_t <- degrees_freedom(m, "t"); tcrit <- qt(0.975, df_t)
  map_dfr(pts_n, function(n0) {
    E0 <- log1p(n0)
    g  <- unname(b[bvar] + b[evar] * E0)
    se <- unname(sqrt(V[bvar, bvar] + (E0^2) * V[evar, evar] + 2 * E0 * V[bvar, evar]))
    pval <- 2 * pt(abs(g / se), df = df_t, lower.tail = FALSE)
    tibble(
      cultura = crop_lbl, n_emb_l1 = n0, ln1p_n = E0,
      pct_obs_abaixo = 100 * mean(d_emb_eff$n_emb_l1 <= n0),
      pct_obs_acima  = 100 * mean(d_emb_eff$n_emb_l1 >= n0),
      gradiente = g, ep = se, ic95_inf = g - tcrit * se, ic95_sup = g + tcrit * se, p = pval
    )
  })
}

In [ ]:
# Reúne as estimativas do gradiente para soja e milho.
grad_df <- bind_rows( calc_grad(B2_s1, "xs", "emb_xs", "soja"), calc_grad(B2_mod_m, "xm", "emb_xm", "milho") )

In [ ]:
# Prepara b_s1 para esta etapa da análise.
b_s1 <- coef(B2_s1)

In [ ]:
# Prepara n_zero_soja para esta etapa da análise.
n_zero_soja <- expm1(-b_s1["xs"] / b_s1["emb_xs"])

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("\n=== Onda 2.5: Embargos — hierarquia Eq. (4) e gradiente marginal Eq. (5) (ponto zero soja N* = %.1f; pct N>=%d = %.1f%%) ===\n",
            n_zero_soja, ceiling(n_zero_soja), 100 * mean(d_emb_eff$n_emb_l1 >= ceiling(n_zero_soja))))

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(B2_s1, B2_area_full_s, B2_mod_m, B2_area_full_m,
    headers = c("Soja N Emb.", "Soja Area Full", "Milho N Emb.", "Milho Area Full"))

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(grad_df), row.names = FALSE)

In [ ]:
# Exporta onda2_embargos_gradiente.csv para a pasta da versão 3.
write.csv(grad_df, file.path(out_dir, "onda2_embargos_gradiente.csv"), row.names = FALSE, fileEncoding = "UTF-8")

### FWL, HAC e Driscoll–Kraay (Tabela A6, Painel A)

In [ ]:
# Calcula a projeção FWL e compara erros-padrão HAC e Driscoll–Kraay.
fwl_hac <- function(crop) {
  xvar <- if (crop == "soja") "xs" else "xm"
  ztr  <- if (crop == "soja") "zs_tr" else "zm_tr"
  m_ref <- if (crop == "soja") M1 else M2
  ry <- resid(feols(as.formula(paste("y_agr ~", ztr, "+ rain | ibge + bioma^ano")), data = d_eff))
  rx <- resid(feols(as.formula(paste(xvar, "~", ztr, "+ rain | ibge + bioma^ano")), data = d_eff))
  beta_fwl <- sum(rx * ry) / sum(rx^2)
  stopifnot(abs(beta_fwl - coef(m_ref)[xvar]) < 1e-10)
  e_fwl <- ry - beta_fwl * rx
  yr_df <- tibble(ano = d_eff$ano, prod_xy = rx * ry, denom = rx^2, score = rx * e_fwl) %>%
    group_by(ano) %>%
    summarise(S_t = sum(prod_xy), D_t = sum(denom), g_t = sum(score), .groups = "drop") %>%
    arrange(ano)
  T_yr <- nrow(yr_df)
  m_sc <- lm(g_t ~ 1, data = yr_df)
  V_nw2 <- NeweyWest(m_sc, lag = 2, prewhite = FALSE)
  se_nw2 <- sqrt(as.numeric(V_nw2) * (T_yr^2) / (sum(yr_df$D_t)^2))
  p_nw2  <- 2 * pt(abs(beta_fwl / se_nw2), df = T_yr - 1, lower.tail = FALSE)
  RES[[length(RES) + 1]] <<- tibble(
    id = paste0("R_fwl_hac_", crop), grupo = "onda2_fwl_hac",
    descricao = paste(crop, "FWL exato 24 scores (Newey-West lag 2)"), amostra = "obs",
    termo = xvar, estimativa = beta_fwl, erro_padrao = se_nw2, p = p_nw2, n = nrow(d_eff),
    r2_total = as.numeric(r2(m_ref, "r2")), r2_within = as.numeric(r2(m_ref, "wr2"))
  )
  m_dk2 <- summary(m_ref, vcov = DK(2) ~ ano)
  rec(m_dk2, paste0("R_dk2_", crop), "onda2_fwl_hac", paste(crop, "Driscoll-Kraay lag 2"), "obs")
  cat(sprintf("FWL-HAC (%-5s): beta = %.4f | EP score-HAC(2) = %.4f (p = %.4f) | EP DK(2) = %.4f (p = %.4f)\n",
              crop, beta_fwl, se_nw2, p_nw2, coeftable(m_dk2)[xvar, 2], coeftable(m_dk2)[xvar, 4]))
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.6: Projeção FWL exata com HAC nos 24 scores e Driscoll-Kraay ===\n")

In [ ]:
# Executa a inferência FWL com HAC e Driscoll–Kraay.
fwl_hac("soja")

In [ ]:
# Executa a inferência FWL com HAC e Driscoll–Kraay.
fwl_hac("milho")

### Tercis de aptidão e resumo de testes

In [ ]:
# Prepara mun_terc para esta etapa da análise.
mun_terc <- mun_v2 %>% mutate(terc_s = ntile(apt_s, 3), terc_m = ntile(apt_m, 3))

In [ ]:
# Prepara as interações de preço e tendência por tercil de aptidão.
d_terc <- d_obs %>%
  left_join(mun_terc %>% select(ibge_int, terc_s, terc_m), by = "ibge_int") %>%
  mutate(
    s_t2 = as.numeric(terc_s == 2), s_t3 = as.numeric(terc_s == 3),
    m_t2 = as.numeric(terc_m == 2), m_t3 = as.numeric(terc_m == 3),
    lps_t2 = lps * s_t2, lps_t3 = lps * s_t3, tr_s_t2 = trend * s_t2, tr_s_t3 = trend * s_t3,
    lpm_t2 = lpm * m_t2, lpm_t3 = lpm * m_t3, tr_m_t2 = trend * m_t2, tr_m_t3 = trend * m_t3
  )

In [ ]:
# Estima ou registra a especificação NL_s.
NL_s <- rec(feols(y_agr ~ lps_t2 + lps_t3 + tr_s_t2 + tr_s_t3 + rain | ibge + bioma^ano, d_terc, vcov = ~ibge),
            "NL_terc_soja", "onda2_nao_linearidade", "Soja: tercis de aptidao GAEZ (ref = T1)", "obs")

In [ ]:
# Estima ou registra a especificação NL_m.
NL_m <- rec(feols(y_agr ~ lpm_t2 + lpm_t3 + tr_m_t2 + tr_m_t3 + rain | ibge + bioma^ano, d_terc, vcov = ~ibge),
            "NL_terc_milho", "onda2_nao_linearidade", "Milho: tercis de aptidao GAEZ (ref = T1)", "obs")

In [ ]:
# Testa diferenças nas interações de preço entre tercis de aptidão.
for (nl in list(list(m = NL_s, v2 = "lps_t2", v3 = "lps_t3", pat = "^lps_t[23]$", crop = "soja"),
                list(m = NL_m, v2 = "lpm_t2", v3 = "lpm_t3", pat = "^lpm_t[23]$", crop = "milho"))) {
  w <- wald(nl$m, keep = nl$pat, print = FALSE)
  add_test("nao_linearidade", "Wald conjunto T2=T3=0 (ref T1)", nl$crop, "obs",
           stat = w$stat, df1 = w$df1, df2 = w$df2, p = w$p)
  b <- coef(nl$m); V <- vcov(nl$m); df_t <- degrees_freedom(nl$m, "t")
  d32 <- b[nl$v3] - b[nl$v2]; se32 <- sqrt(V[nl$v3, nl$v3] + V[nl$v2, nl$v2] - 2 * V[nl$v3, nl$v2])
  p32 <- 2 * pt(abs(d32 / se32), df = df_t, lower.tail = FALSE)
  add_test("nao_linearidade", "Contraste T3 - T2", nl$crop, "obs",
           est = d32, ep = se32, stat = d32 / se32, df2 = df_t, p = p32)
}

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Onda 2.6b: Não-linearidade por tercis de aptidão GAEZ (ref = Tercil 1) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(NL_s, NL_m, headers = c("Soja Tercis Z", "Milho Tercis Z"))

In [ ]:
# Prepara tests_df para esta etapa da análise.
tests_df <- bind_rows(TESTS_O2)

In [ ]:
# Exporta onda2_testes_conjuntos.csv para a pasta da versão 3.
write.csv(tests_df, file.path(out_dir, "onda2_testes_conjuntos.csv"), row.names = FALSE, fileEncoding = "UTF-8")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Resumo de Testes Conjuntos e Contrastes (Onda 2) ===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(tests_df), row.names = FALSE)

***

## 2.8c Tabelas A4, A5 e A6 — Pecuária, sensibilidade dos embargos e ajuste de Holm–Bonferroni

- **Pecuária e conversão para pastagem:** modelos complementares usando o preço real internacional da carne bovina (`carne_bovina`, lag1) e a razão *crop-to-beef* interagidos com ocupação prévia de pastagem (1985–2000), aptidão mecanizável GAEZ e fração amazônica.
- **H1b (ponte fiscal + sensibilidade à focalização tática):** completa a ponte alinhada na subamostra fiscal ($N = 1.135$ / $1.125$ efetivos, mesmos FEs e tendência) e testa sensibilidade dos embargos do IBAMA ao controle dinâmico por conversão defasada ($Y_{m,t-1}$) e placebo futuro de embargos ($E_{m,t+1}$).
- **Multiplicidade:** cálculo formal dos *p*-valores ajustados por **Holm--Bonferroni** dentro de famílias pré-especificadas.


In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("=== INICIANDO SEÇÃO 2.8c (FASE 2) ===\n")

### Tabela A4 — Conversão para pastagem e agricultura sob preços da carne bovina e razão *crop-to-beef*, 2001–2024

In [ ]:
# Prepara o preço defasado da carne bovina na grade municipal.
pb_tbl <- raw_v2 %>% transmute( ibge_int = as.integer(z0_codigo_ibge), ano = as.integer(z0_ano),
    pb_l1 = as.numeric(x2a_preco_real_usd2024_media_anual_lag1__carne_bovina) )

In [ ]:
# Prepara anos_pb para esta etapa da análise.
anos_pb <- pb_tbl %>% distinct(ano, pb_l1) %>% arrange(ano)

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(nrow(anos_pb) == 24, !anyNA(anos_pb$pb_l1))

In [ ]:
# Prepara c_pb para esta etapa da análise.
c_pb <- mean(log(anos_pb$pb_l1))

In [ ]:
# Monta o painel para pecuária e preços relativos.
d_pas2 <- d_camA %>%
  left_join(pb_tbl, by = c("ibge_int", "ano")) %>%
  mutate(
    lpb = log(pb_l1) - c_pb,
    lcb_s = lps - lpb,
    lcb_m = lpm - lpb,
    zpas_tr = z_exp_pas * trend,
    amz_tr  = amz * trend,
    x_pb_zpas = lpb * z_exp_pas,
    x_pb_zs   = lpb * zs,
    x_pb_zm   = lpb * zm,
    x_pb_amz  = lpb * amz,
    x_cb_s_zs = lcb_s * zs,
    x_cb_m_zm = lcb_m * zm
  )

In [ ]:
# Estima ou registra a especificação PAS_1.
PAS_1 <- rec(feols(y_pas ~ x_pb_zpas + zpas_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "PAS_beef_zpas", "fase2_pecuaria", "Pastagem: preco boi x share pastagem 1985-2000", "obs")

In [ ]:
# Estima ou registra a especificação PAS_2.
PAS_2 <- rec(feols(y_pas ~ x_pb_zs + zs_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "PAS_beef_zs", "fase2_pecuaria", "Pastagem: preco boi x aptidao GAEZ soja", "obs")

In [ ]:
# Estima ou registra a especificação PAS_3.
PAS_3 <- rec(feols(y_pas ~ x_cb_s_zs + zs_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "PAS_c2b_soja", "fase2_pecuaria", "Pastagem: razao soja/boi x GAEZ soja", "obs")

In [ ]:
# Estima ou registra a especificação PAS_4.
PAS_4 <- rec(feols(y_pas ~ xs + x_pb_zs + zs_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "PAS_both_soja", "fase2_pecuaria", "Pastagem: soja e boi simultaneos x GAEZ soja", "obs")

In [ ]:
# Estima ou registra a especificação PAS_5.
PAS_5 <- rec(feols(y_pas ~ xm + x_pb_zm + zm_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "PAS_both_milho", "fase2_pecuaria", "Pastagem: milho e boi simultaneos x GAEZ milho", "obs")

In [ ]:
# Estima ou registra a especificação PAS_6.
PAS_6 <- rec(feols(y_pas ~ x_pb_amz + amz_tr + rain | ibge + ano, d_pas2, vcov = ~ibge),
             "PAS_beef_amz", "fase2_pecuaria", "Pastagem: preco boi x share Amazonia (FE ano)", "obs")

In [ ]:
# Estima ou registra a especificação AGR_1.
AGR_1 <- rec(feols(y_agr ~ x_cb_s_zs + zs_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "AGR_c2b_soja", "fase2_pecuaria", "Agricultura: razao soja/boi x GAEZ soja", "obs")

In [ ]:
# Estima ou registra a especificação AGR_2.
AGR_2 <- rec(feols(y_agr ~ xs + x_pb_zs + zs_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "AGR_both_soja", "fase2_pecuaria", "Agricultura: soja e boi simultaneos x GAEZ soja", "obs")

In [ ]:
# Estima ou registra a especificação AGR_3.
AGR_3 <- rec(feols(y_agr ~ x_pb_zpas + zpas_tr + rain | ibge + bioma^ano, d_pas2, vcov = ~ibge),
             "AGR_beef_zpas", "fase2_pecuaria", "Agricultura: preco boi x share pastagem 1985-2000", "obs")

In [ ]:
# Calcula o teste de hipótese w_pas.
w_pas <- wald(PAS_4, keep = "^(xs|x_pb_zs)$", print = FALSE)

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat(sprintf("Wald conjunto Pastagem (xs = x_pb_zs = 0): F = %.3f (p = %.4f)\n", w_pas$stat, w_pas$p))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Fase 2.A: Pastagem e Agricultura com Preço da Carne Bovina e Razão Crop-to-Beef ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(PAS_1, PAS_2, PAS_3, PAS_4, PAS_6, AGR_1, AGR_2,
    headers = c("Pas Boi×Zpas", "Pas Boi×Zgaez", "Pas Soja/Boi", "Pas Soja+Boi", "Pas Boi×Amz", "Agr Soja/Boi", "Agr Soja+Boi"))

In [ ]:
# Prepara pas_df para esta etapa da análise.
pas_df <- bind_rows(RES) %>% filter(grupo == "fase2_pecuaria" | id %in% c("R_pas_soja", "R_pas_milho"))

In [ ]:
# Exporta fase2_pastagem_pecuaria.csv para a pasta da versão 3.
write.csv(pas_df, file.path(out_dir, "fase2_pastagem_pecuaria.csv"), row.names = FALSE, fileEncoding = "UTF-8")

### Tabela A5 — Ponte na subamostra fiscal alinhada ($N=1.125$) e testes de sensibilidade à focalização dos embargos

In [ ]:
# Estima ou registra a especificação F_emb_mod_sub.
F_emb_mod_sub <- rec(feols(y_agr ~ xs + emb_esp_l1 + emb_ps + emb_zs + emb_xs + zs_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                     "F_emb_mod_sub_soja", "fase2_h1b_alinhado", "Embargos N na amostra fiscal alinhada (soja)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_m6_al_m.
F_m6_al_m     <- rec(feols(y_agr ~ xm + f1 + zm_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                     "F_M6_alinhado_milho", "fase2_h1b_alinhado", "M6 milho alinhado (ibge + bioma^ano + zm_tr)", "fiscal")

In [ ]:
# Estima ou registra a especificação F_emb_sub_m.
F_emb_sub_m   <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_xm + zm_tr + rain | ibge + bioma^ano, d_fisc_al, vcov = ~ibge),
                     "F_emb_area_sub_milho", "fase2_h1b_alinhado", "Embargos area na amostra fiscal alinhada (milho)", "fiscal")

In [ ]:
# Calcula a conversão observada em t−1 sem preencher valores ausentes.
y_lag_tbl <- df2 %>% arrange(ibge_int, ano) %>% group_by(ibge_int) %>%
  mutate(y_agr_l1 = ifelse(lag(obs) & (ano - lag(ano) == 1L), lag(y_agr), NA_real_)) %>% ungroup() %>%
  select(ibge_int, ano, y_agr_l1)

In [ ]:
# Aplica setorder no ponto correspondente da análise.
setorder(emb_panel, ibge_int, yr)

In [ ]:
# Prepara o placebo com registros de embargo em t+1.
emb_lead_tbl <- as_tibble(emb_panel[, .(
  ibge_int,
  ano = yr,
  emb_esp_f1  = log1p(shift(n_emb_esp, 1L, type = "lead")),
  emb_area_f1 = log1p(shift(area_emb_esp_ha, 1L, type = "lead"))
), by = ibge_int][, .(ibge_int, ano, emb_esp_f1, emb_area_f1)])

In [ ]:
# Une as variáveis defasadas e futuras para os testes de sensibilidade.
d_emb_sens <- d_camB2 %>% left_join(y_lag_tbl, by = c("ibge_int", "ano")) %>%
  left_join(emb_lead_tbl, by = c("ibge_int", "ano")) %>% mutate( emb_ps_f1 = lps * emb_esp_f1,
    emb_zs_f1 = zs * emb_esp_f1, emb_xs_f1 = xs * emb_esp_f1 )

In [ ]:
# Estima ou registra a especificação DYN_1.
DYN_1 <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_xs + y_agr_l1 + zs_tr + rain | ibge + bioma^ano, d_emb_sens, vcov = ~ibge),
             "B2_dyn_area_soja", "fase2_h1b_sens", "Soja + area embargada + Y_{t-1}", "obs c/ lag")

In [ ]:
# Estima ou registra a especificação DYN_2.
DYN_2 <- rec(feols(y_agr ~ xm + emb_area_l1 + emb_area_xm + y_agr_l1 + zm_tr + rain | ibge + bioma^ano, d_emb_sens, vcov = ~ibge),
             "B2_dyn_area_milho", "fase2_h1b_sens", "Milho + area embargada + Y_{t-1}", "obs c/ lag")

In [ ]:
# Estima ou registra a especificação DYN_3.
DYN_3 <- rec(feols(y_agr ~ xs + emb_esp_l1 + emb_ps + emb_zs + emb_xs + y_agr_l1 + zs_tr + rain | ibge + bioma^ano, d_emb_sens, vcov = ~ibge),
             "B2_dyn_mod_soja", "fase2_h1b_sens", "Soja + moderacao embargos + Y_{t-1}", "obs c/ lag")

In [ ]:
# Estima ou registra a especificação LEAD_1.
LEAD_1 <- rec(feols(y_agr ~ xs + emb_area_l1 + emb_area_f1 + emb_area_xs + zs_tr + rain | ibge + bioma^ano, d_emb_sens, vcov = ~ibge),
              "B2_lead_area_soja", "fase2_h1b_sens", "Soja + area embargada t-1 e placebo t+1", "obs c/ lead")

In [ ]:
# Estima ou registra a especificação LEAD_2.
LEAD_2 <- rec(feols(y_agr ~ xs + emb_esp_l1 + emb_ps + emb_zs + emb_xs + emb_esp_f1 + emb_ps_f1 + emb_zs_f1 + emb_xs_f1 + zs_tr + rain | ibge + bioma^ano, d_emb_sens, vcov = ~ibge),
              "B2_lead_mod_soja", "fase2_h1b_sens", "Soja + moderacao embargos t-1 e placebo t+1", "obs c/ lead")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Fase 2.B: Ponte Fiscal Alinhada e Sensibilidade a Focalização (Y_{t-1} e E_{t+1}) ===\n")

In [ ]:
# Exibe coeficientes, erros-padrão e métricas de ajuste.
tab(F_emb_mod_sub, F_m6_al_m, F_emb_sub_m, DYN_1, DYN_3, LEAD_1, LEAD_2,
    headers = c("EmbMod Fisc", "M6 Al. Milho", "EmbArea Milho", "Dyn Area Soja", "Dyn Mod Soja", "Lead Area", "Lead Mod"))

In [ ]:
# Prepara emb_sens_df para esta etapa da análise.
emb_sens_df <- bind_rows(RES) %>% filter(grupo %in% c("fase2_h1b_alinhado", "fase2_h1b_sens", "onda2_pontes") & grepl("^(F_|B2_)", id))

In [ ]:
# Exporta fase2_embargos_sensibilidade.csv para a pasta da versão 3.
write.csv(emb_sens_df, file.path(out_dir, "fase2_embargos_sensibilidade.csv"), row.names = FALSE, fileEncoding = "UTF-8")

### Tabela A6 — Inferência alternativa, painel balanceado, contraste entre destinos e ajuste de Holm–Bonferroni

In [ ]:
# Prepara res_tmp para esta etapa da análise.
res_tmp <- bind_rows(RES)

In [ ]:
# Vincula cada hipótese aos resultados exportados do modelo correspondente.
pick_p <- function(familia, id_vec, termo_vec, papel_vec) {
  tibble(familia = familia, id = id_vec, termo = termo_vec, papel = papel_vec) %>%
    left_join(res_tmp %>% select(id, termo, estimativa, erro_padrao, p, n) %>% distinct(id, termo, .keep_all = TRUE),
              by = c("id", "termo"))
}

In [ ]:
# Prepara s_amz_id para esta etapa da análise.
s_amz_id <- paste0("S_soja_", substr(bioma_amz, 1, 3))

In [ ]:
# Prepara s_cer_id para esta etapa da análise.
s_cer_id <- paste0("S_soja_", substr(bioma_cer, 1, 3))

In [ ]:
# Prepara m_amz_id para esta etapa da análise.
m_amz_id <- paste0("S_milho_", substr(bioma_amz, 1, 3))

In [ ]:
# Prepara m_cer_id para esta etapa da análise.
m_cer_id <- paste0("S_milho_", substr(bioma_cer, 1, 3))

In [ ]:
# Monta as famílias de testes e ajusta os p-valores por Holm–Bonferroni.
holm_tbl <- bind_rows(
  pick_p("F1_H1a_principais_biomas",
         c("M1", "M2", s_amz_id, s_cer_id, m_amz_id, m_cer_id),
         c("xs", "xm", "xs", "xs", "xm", "xm"),
         c("Soja MT", "Milho MT", "Soja Amazonia", "Soja Cerrado", "Milho Amazonia", "Milho Cerrado")),
  pick_p("F2_H1b_instituicoes",
         c("B2_emb_area_soja", "B2_emb_area_milho", "B2_emb_mod_soja", "B2_emb_mod_milho", "F_M6_alinhado_soja", "F_M7_alinhado_soja"),
         c("emb_area_l1", "emb_area_l1", "emb_xs", "emb_xm", "f1", "f1_xs"),
         c("Area emb. soja", "Area emb. milho", "Mod. N emb. soja", "Mod. N emb. milho", "Autos alinhado soja", "Mod. autos alinhado soja")),
  pick_p("F3_Regimes_2006_11_e_Destinos",
         c("REG_prc_soja_Amz", "REG_prc_soja_Cer", "REG_lvl_soja_Amz", "REG_lvl_soja_Cer", "STACK_soja", "STACK_milho"),
         c("xs_p2", "xs_p2", "xs_r_p2", "xs_r_p2", "xs_agr", "xm_agr"),
         c("Painel A Amz 06-11", "Painel A Cer 06-11", "Painel B Amz 06-11", "Painel B Cer 06-11", "Empilhado Agr-Pas soja", "Empilhado Agr-Pas milho")),
  pick_p("F4_Pecuaria_Pastagem",
         c("PAS_beef_zpas", "PAS_beef_zs", "PAS_c2b_soja", "PAS_both_soja", "PAS_both_soja", "R_pas_soja", "R_pas_milho"),
         c("x_pb_zpas", "x_pb_zs", "x_cb_s_zs", "xs", "x_pb_zs", "xs", "xm"),
         c("Pas Boi x Zpas", "Pas Boi x GAEZ", "Pas Soja/Boi x GAEZ", "Pas conjunta (Soja)", "Pas conjunta (Boi)", "Pas Soja x GAEZ", "Pas Milho x GAEZ"))
) %>%
  group_by(familia) %>%
  mutate(m_testes = n(), p_holm = p.adjust(p, method = "holm")) %>%
  ungroup()

In [ ]:
# Confere as condições necessárias para seguir com a análise.
stopifnot(!anyNA(holm_tbl$p_holm))

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== Fase 2.C: Multiplicidade Holm-Bonferroni por Família ===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(holm_tbl), row.names = FALSE)

In [ ]:
# Exporta fase2_holm_bonferroni.csv para a pasta da versão 3.
write.csv(holm_tbl, file.path(out_dir, "fase2_holm_bonferroni.csv"), row.names = FALSE, fileEncoding = "UTF-8")

***

## 2.9 Consolidação Geral, Exportação para Auditoria e Veredito (Etapa 15)

Exporta todos os modelos estimados nas seções 2.2 a 2.8c para `2_INTI/audit/modelos_h1_prec_v3_20261118/saidas/resultados_modelos.csv` e avalia os 6 critérios pré-especificados em `PRE_ESPECIFICACAO.md`.

### Consolidação dos coeficientes

In [ ]:
# Consolida o registro com todas as estimações e testes.
res_all <- bind_rows(RES)

In [ ]:
# Exporta resultados_modelos.csv para a pasta da versão 3.
write.csv(res_all, file.path(out_dir, "resultados_modelos.csv"), row.names = FALSE, fileEncoding = "UTF-8")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("Total exportado para resultados_modelos.csv:", n_distinct(res_all$id), "modelos/testes;", nrow(res_all), "linhas.\n")

### Critérios pré-especificados

In [ ]:
# Seleciona o resultado de um modelo e termo específicos.
get1 <- function(id, termo) res_all %>% filter(id == !!id, termo == !!termo) %>% slice(1)

In [ ]:
# Aplica os critérios pré-especificados à cultura selecionada.
veredito <- function(crop) {
  x <- if (crop == "soja") "xs" else "xm"; xf <- paste0(x, "_f")
  pri <- get1(ifelse(crop == "soja", "M1", "M2"), x)
  cl2 <- get1(paste0("R_cl2_", crop), x)
  tws <- get1(paste0("R_2step_", crop), "lp")
  lo  <- res_all %>% filter(grepl(paste0("^R_loyo_", crop, "_"), id), termo == x)
  pl  <- get1(paste0("R_plac_", crop), xf)
  w06 <- get1(paste0("R_w06_", crop), x)
  crit <- tibble(
    criterio = c("C0 principal: estimativa > 0 e p < 0,10",
                 "C1 cluster município+ano: mesmo sinal e p < 0,10",
                 "C2 regressão em 24 pontos: mesmo sinal e p < 0,10",
                 "C3 exclusão de um ano: mesmo sinal em 24 e p < 0,10 em >= 19",
                 "C4 placebo: preço futuro com p >= 0,10",
                 "C5 2006-2024: mesmo sinal"),
    atendido = c(pri$estimativa > 0 & pri$p < 0.10,
                 sign(cl2$estimativa) == sign(pri$estimativa) & cl2$p < 0.10,
                 sign(tws$estimativa) == sign(pri$estimativa) & tws$p < 0.10,
                 all(sign(lo$estimativa) == sign(pri$estimativa)) & sum(lo$p < 0.10) >= 19,
                 pl$p >= 0.10,
                 sign(w06$estimativa) == sign(pri$estimativa)),
    valor = c(sprintf("est %.4f p %.3f", pri$estimativa, pri$p),
              sprintf("est %.4f p %.3f", cl2$estimativa, cl2$p),
              sprintf("est %.4f p %.3f", tws$estimativa, tws$p),
              sprintf("sinais iguais %d/24; p<0,10 em %d", sum(sign(lo$estimativa) == sign(pri$estimativa)), sum(lo$p < 0.10)),
              sprintf("est %.4f p %.3f", pl$estimativa, pl$p),
              sprintf("est %.4f p %.3f", w06$estimativa, w06$p))
  )
  classe <- if (!crit$atendido[1]) "Não consistente" else if (all(crit$atendido)) "Consistente" else "Parcial"
  list(crit = crit, classe = classe)
}

In [ ]:
# Calcula o veredito pré-especificado para soja.
vs <- veredito("soja")

In [ ]:
# Calcula o veredito pré-especificado para milho.
vm <- veredito("milho")

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== VEREDITO PRÉ-ESPECIFICADO — SOJA (M1):", vs$classe, "===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(vs$crit), row.names = FALSE)

In [ ]:
# Exibe o resultado ou identifica a saída desta etapa.
cat("\n=== VEREDITO PRÉ-ESPECIFICADO — MILHO (M2):", vm$classe, "===\n")

In [ ]:
# Exibe os resultados calculados nesta etapa.
print(as.data.frame(vm$crit), row.names = FALSE)

In [ ]:
# Exporta veredito_criterio.csv para a pasta da versão 3.
write.csv(bind_rows(mutate(vs$crit, cultura = "soja", classe = vs$classe), mutate(vm$crit, cultura = "milho", classe = vm$classe)),
          file.path(out_dir, "veredito_criterio.csv"), row.names = FALSE, fileEncoding = "UTF-8")